In [ ]:
# Sentinel-3A SP3 download (CDDIS, correct v30 paths, TAI→UTC corrected)
import requests, gzip, subprocess, tempfile, os
from datetime import datetime, timedelta, timezone
import numpy as np

CACHE_DIR = os.environ.get('NTN_CACHE_DIR', '/content' if os.path.isdir('/content') else './data_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

TOKEN = os.environ.get("EARTHDATA_TOKEN") or input("EARTHDATA_TOKEN: ")
headers = {'Authorization': f'Bearer {TOKEN}'}


sp3_files = [
    f"ssas3a30.b{b}.e{e}.DG_.sp3.001.Z"
    for b, e in [
        ("24336","24346"), ("24345","24355"), ("24354","24364"), ("24363","25008"),
        ("25006","25016"), ("25015","25025"), ("25024","25034"), ("25033","25043"),
        ("25042","25052"), ("25051","25061"), ("25060","25070"), ("25069","25079"),
        ("25078","25088"), ("25087","25097"), ("25096","25106"), ("25105","25115"),
        ("25114","25124"), ("25123","25133"), ("25132","25142"), ("25141","25151"),
        ("25150","25160"), ("25159","25169"), ("25168","25178"), ("25177","25187"),
        ("25186","25196"), ("25195","25205"), ("25204","25214"), ("25213","25223"),
        ("25222","25232"), ("25231","25241"), ("25240","25250"), ("25249","25259"),
        ("25258","25268"), ("25267","25277"), ("25276","25286"), ("25285","25295"),
        ("25294","25304"), ("25303","25313"), ("25312","25322"), ("25321","25331"),
        ("25330","25340"), ("25339","25349"),
    ]
]

all_tj, all_pos = [], []
for fn in sp3_files:
    url = f"https://cddis.nasa.gov/archive/doris/products/orbits/ssa/s3a/{fn}"
    print(f"Downloading {fn}...", end=" ")
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code != 200:
        print(f"HTTP {resp.status_code}"); continue

    with tempfile.NamedTemporaryFile(suffix='.Z', delete=False) as tmp:
        tmp.write(resp.content)
        tmp_path = tmp.name
    try:
        subprocess.run(['uncompress', '-f', tmp_path], check=True)
        sp3_path = tmp_path[:-2]
        with open(sp3_path) as f:
            text = f.read()
        os.unlink(sp3_path)
    except:
        try:
            text = gzip.decompress(resp.content).decode('utf-8', errors='ignore')
        except:
            text = resp.content.decode('utf-8', errors='ignore')
        if os.path.exists(tmp_path): os.unlink(tmp_path)

    current_epoch = None; pts = 0
    for line in text.split('\n'):
        line = line.rstrip()
        if not line: continue
        if line.startswith('*'):
            parts = line[1:].split()
            if len(parts) >= 6:
                try:
                    yr,mo,dy = int(parts[0]),int(parts[1]),int(parts[2])
                    hr,mi = int(parts[3]),int(parts[4])
                    sec = float(parts[5])
                    dt = datetime(yr,mo,dy,hr,mi,int(sec),tzinfo=timezone.utc)
                    jd = 2451545.0 + (dt - datetime(2000,1,1,12,0,0,tzinfo=timezone.utc)).total_seconds()/86400
                    current_epoch = jd
                except: pass
        elif line.startswith('P') and current_epoch:
            sid = line[1:4].strip()
            if sid == 'L74':
                try:
                    x,y,z = float(line[4:18]), float(line[18:32]), float(line[32:46])
                    if abs(x) < 99999:
                        all_tj.append(current_epoch)
                        all_pos.append([x, y, z])
                        pts += 1
                except: pass
    print(f"{pts} points")

if all_tj:
    tj = np.array(all_tj); pos = np.array(all_pos)
    _, uniq = np.unique(tj, return_index=True)
    tj, pos = tj[uniq], pos[uniq]

   
    tj = tj - 37.0 / 86400.0

    alt = np.mean(np.linalg.norm(pos, axis=1)) - 6378.137
    print(f"\nTotal: {len(tj)} Sentinel-3A points, alt={alt:.0f} km")
    print("TAI→UTC correction applied (-37 seconds)")

    np.save(os.path.join(CACHE_DIR, 's3a_tj.npy'), tj)
    np.save(os.path.join(CACHE_DIR, 's3a_pos.npy'), pos)
    print(f"Saved to {CACHE_DIR}/s3a_tj.npy and {CACHE_DIR}/s3a_pos.npy")
else:
    print("No data extracted")

In [ ]:

!pip install sgp4 torch numpy requests scipy matplotlib viresclient pandas astropy -q

import os, json, time, math, getpass, warnings, hashlib
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field
from typing import List, Optional, Dict
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import requests
from viresclient import SwarmRequest, set_token
from scipy.interpolate import CubicSpline

import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['mathtext.fontset'] = 'dejavusans'
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sgp4.api import Satrec, WGS72, jday
import tarfile, gzip, io

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('results', exist_ok=True)
CACHE_DIR = os.environ.get('NTN_CACHE_DIR', '/content' if os.path.isdir('/content') else './data_cache')
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Device: {device}')
print(f'Data cache: {CACHE_DIR}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  CUDA: {torch.version.cuda}, torch: {torch.__version__}')

START_DATE = "2024-12-01"
END_DATE   = "2025-12-01"
TRAIN_END  = "2025-09-01"
TEST_START = "2025-09-01"

EARTHDATA_TOKEN = os.environ.get('EARTHDATA_TOKEN', '')
VIRES_TOKEN     = os.environ.get('VIRES_TOKEN', '')
if not EARTHDATA_TOKEN:
    EARTHDATA_TOKEN = getpass.getpass('EARTHDATA_TOKEN: ')
if not VIRES_TOKEN:
    VIRES_TOKEN = getpass.getpass('VIRES_TOKEN: ')

# ══════════════════════════════════════════════════════════════════════
# 3GPP NR-NTN BAND DEFINITIONS (n255, n256)
# Per 3GPP TS 38.101-5 (NR-NTN UE RF) and TR 38.821 (NTN scenarios)
# n255: L-band NTN    UL 1626.5-1660.5 MHz    DL 1525-1559 MHz
# n256: S-band NTN    UL 1980-2010   MHz    DL 2170-2200 MHz
# ══════════════════════════════════════════════════════════════════════
NTN_BANDS = {
    'n255': {
        'ul_mhz_center': 1643.5,   # 1626.5-1660.5
        'dl_mhz_center': 1542.0,   # 1525.0-1559.0
        'family': 'L-band',
    },
    'n256': {
        'ul_mhz_center': 1995.0,   # 1980-2010
        'dl_mhz_center': 2185.0,   # 2170-2200
        'family': 'S-band',
    },
}
# For the primary Doppler loss during training, use n256 DL (highest freq
# -> hardest case). 
PRIMARY_CARRIER_HZ = NTN_BANDS['n256']['dl_mhz_center'] * 1e6

# ══════════════════════════════════════════════════════════════════════
# 3GPP REFERENCE NTN LINK-BUDGET CONSTANTS
# Sources: TR 38.821 v16.1.0, TS 36.211, TS 36.213, TS 38.211
# ══════════════════════════════════════════════════════════════════════
NTN_3GPP = {
    # PRACH Doppler tolerance (post-pre-compensation residual)
    # Derived from NPRACH preamble design in TS 36.211 and IoT-NTN analysis.
    # This is an upper bound on residual Doppler after UE pre-compensates
    # using ephemeris. Reference: TR 36.763 IoT-NTN study.
    'prach_doppler_budget_hz': 950.0,

    # NR PRACH residual Doppler tolerance is tighter; SCS-dependent.
    # For NR-NTN with 15 kHz SCS (TS 38.211), residual target ~<=100 Hz.
    'nr_prach_doppler_budget_hz': 100.0,

    # Cyclic prefix length (seconds) at 15 kHz SCS, normal CP, for NR.
    'nr_cp_normal_15khz_sec': 4.69e-6,   # Ts = 1/(15000*2048) * 144 samples
    'nr_cp_extended_15khz_sec': 16.67e-6,

    # Timing advance granularity (TS 38.213). NTN uses common TA based on
    # ephemeris. Error beyond CP length causes ISI / sync failure.
    'ta_tolerance_sec': 4.69e-6,

    # Reference beamwidth (HPBW) for NTN beam pointing analysis.
    # TR 38.821 IoT-NTN Set-1: ~4.41 deg HPBW.
    'ntn_beamwidth_hpbw_deg': 4.41,

   
    'ref_scenario': 'LEO-600 Set-5 (IoT-NTN)',
    'ref_alt_km': 600.0,
}

print(f'\n3GPP NR-NTN bands configured: n255 ({NTN_BANDS["n255"]["family"]}), '
      f'n256 ({NTN_BANDS["n256"]["family"]})')
print(f'Primary carrier for training loss: n256 DL = '
      f'{PRIMARY_CARRIER_HZ/1e9:.3f} GHz')
print(f'Reference scenario: {NTN_3GPP["ref_scenario"]}')

@dataclass
class Config:
    prop_times: List[int] = field(default_factory=lambda: [10,30,60,90,120,180,240,300,360,720,1080,1440])
    obs_per_sample: int = 5
    r_norm: float = 6878.0; v_norm: float = 7.5
    t_norm: float = 1440.0
    age_norm: float = 168.0
    hidden: int = 48; layers: int = 3; epochs: int = 30
    lr: float = 1e-3; lr_min: float = 1e-5; batch_size: int = 128
    vel_weight: float = 1.0
    rr_weight:  float = 1.0
    dop_weight: float = 1.0
    sat_embed_dim: int = 4
    
    gate_hidden: int = int(os.environ.get('NTN_GATE_HIDDEN', '24'))
    gate_epochs: int = int(os.environ.get('NTN_GATE_EPOCHS', '8'))
    gate_lr: float = float(os.environ.get('NTN_GATE_LR', '3e-4'))
    gate_min_pos_alpha: float = float(os.environ.get('NTN_GATE_MIN_POS_ALPHA', '0.0'))
    gate_min_vel_alpha: float = float(os.environ.get('NTN_GATE_MIN_VEL_ALPHA', '0.25'))
    max_pos_correction_km: float = float(os.environ.get('NTN_MAX_POS_CORR_KM', '2.5'))
    max_vel_correction_kms: float = float(os.environ.get('NTN_MAX_VEL_CORR_KMS', '0.012'))
    pos_safety_margin_km: float = float(os.environ.get('NTN_POS_SAFETY_MARGIN_KM', '0.02'))
    delay_safety_margin_us: float = float(os.environ.get('NTN_DELAY_SAFETY_MARGIN_US', '0.03'))
    gate_safety_weight: float = float(os.environ.get('NTN_GATE_SAFETY_WEIGHT', '1.0'))
    gate_delay_safety_weight: float = float(os.environ.get('NTN_GATE_DELAY_SAFETY_WEIGHT', '0.25'))
    gate_l1_weight: float = float(os.environ.get('NTN_GATE_L1_WEIGHT', '0.005'))
    low_error_gate_threshold_km: float = float(os.environ.get('NTN_LOW_ERR_THRESHOLD_KM', '0.3'))
    c_km_s: float = 299792.458
    f_dl: float = PRIMARY_CARRIER_HZ   # n256 DL by default
    omega_e: float = 7.2921159e-5; re: float = 6378.137
   
    wgs84_f: float = 1.0 / 298.257223563
    min_eval_elev_deg: float = 10.0      
    visibility_cuts_deg: tuple = (-90.0, 0.0, 10.0, 30.0)
    use_astropy_frames: bool = True      
  
    paper_mode: bool = os.environ.get('NTN_PAPER_MODE', '1') != '0'
    allow_gmst_fallback: bool = os.environ.get('NTN_ALLOW_GMST_FALLBACK', '0') == '1'
    
    iers_file: str = os.environ.get('NTN_IERS_FILE', '')
    iers_auto_download: bool = os.environ.get('NTN_IERS_AUTO_DOWNLOAD', '0') == '1'
    bootstrap_n: int = int(os.environ.get('NTN_BOOTSTRAP_N', '1000'))
    bootstrap_seed: int = int(os.environ.get('NTN_BOOTSTRAP_SEED', '12345'))
    seed: int = 42

cfg = Config()

SATELLITES = {
    'Swarm-A':     {'norad': 39452, 'collection': 'SW_OPER_MODA_SC_1B', 'source': 'vires',  'alt_km': 462},
    'Swarm-B':     {'norad': 39451, 'collection': 'SW_OPER_MODB_SC_1B', 'source': 'vires',  'alt_km': 510},
    'Swarm-C':     {'norad': 39453, 'collection': 'SW_OPER_MODC_SC_1B', 'source': 'vires',  'alt_km': 462},
    'GRACE-FO-1':  {'norad': 43476, 'source': 'podaac', 'alt_km': 490},
    'Sentinel-3A': {'norad': 41335, 'source': 'cddis',  'alt_km': 814},
}

print(f'\nv8 setup complete. {len(SATELLITES)} satellites, '
      f'{len(NTN_BANDS)} NR-NTN bands, paper_mode={cfg.paper_mode}, ready to train.')

In [ ]:
# ── 1. Download Precise Orbits (multi-satellite) ─────────────────────
set_token("https://vires.services/ows",
          token=VIRES_TOKEN, set_default=True)

def datetime_to_jd(dt):
    return 2451545.0 + (dt - datetime(2000,1,1,12,0,0,tzinfo=timezone.utc)).total_seconds()/86400

# ── Swarm download (VirES) ──
def download_swarm(sat_letter, start, end, sampling="PT30S"):
    collection = SATELLITES[f'Swarm-{sat_letter}']['collection']
    print(f'  Swarm-{sat_letter}...')
    s, e = datetime.strptime(start,"%Y-%m-%d"), datetime.strptime(end,"%Y-%m-%d")
    all_dfs = []; cur = s
    while cur < e:
        nx = cur.replace(month=cur.month+1) if cur.month<12 else cur.replace(year=cur.year+1,month=1)
        ce = min(nx, e)
        try:
            req = SwarmRequest("https://vires.services/ows")
            req.set_collection(collection)
            req.set_products(measurements=[],auxiliaries=[],sampling_step=sampling)
            data = req.get_between(cur.strftime("%Y-%m-%dT%H:%M:%S"),ce.strftime("%Y-%m-%dT%H:%M:%S"))
            df = data.as_dataframe()
            if len(df)>0: all_dfs.append(df); print(f'    {cur.strftime("%Y-%m")}: {len(df)} pts')
        except Exception as ex: print(f'    {cur.strftime("%Y-%m")}: FAILED ({ex})')
        cur = ce; time.sleep(1)
    if not all_dfs: return None,None,None
    import pandas as pd
    df = pd.concat(all_dfs).sort_index(); df = df[~df.index.duplicated(keep='first')]
    lr, lo = np.radians(df['Latitude'].values), np.radians(df['Longitude'].values)
    r = df['Radius'].values/1000
    pos = np.column_stack([r*np.cos(lr)*np.cos(lo), r*np.cos(lr)*np.sin(lo), r*np.sin(lr)])
    ts = df.index.to_pydatetime()
    tj = np.array([datetime_to_jd(t.replace(tzinfo=timezone.utc) if t.tzinfo is None else t) for t in ts])
    dt_s = np.diff(tj)*86400; vel = np.zeros_like(pos)
    for i in range(1,len(pos)-1): vel[i]=(pos[i+1]-pos[i-1])/((tj[i+1]-tj[i-1])*86400)
    vel[0]=(pos[1]-pos[0])/dt_s[0]; vel[-1]=(pos[-1]-pos[-2])/dt_s[-1]
    print(f'    Total: {len(df)} pts, r={np.mean(r):.1f} km')
    return tj, pos, vel

# ── GRACE-FO download (PODAAC GNV1B) ──
def download_grace_fo(start, end):
    print(f'  GRACE-FO-1...')
    headers = {'Authorization': f'Bearer {EARTHDATA_TOKEN}'}
    cmr_url = ("https://cmr.earthdata.nasa.gov/search/granules.json"
               f"?short_name=GRACEFO_L1B_ASCII_GRAV_JPL_RL04"
               f"&temporal={start}T00:00:00Z,{end}T00:00:00Z&page_size=40")
    resp = requests.get(cmr_url, headers=headers, timeout=30)
    if resp.status_code != 200:
        print(f'    CMR failed: {resp.status_code}'); return None, None, None
    entries = resp.json().get('feed', {}).get('entry', [])
    tgz_urls = sorted(set(
        link['href'] for entry in entries for link in entry.get('links', [])
        if 'noLRI.tgz' in link.get('href', '') and link['href'].startswith('https://archive.podaac')))
    print(f'    Found {len(tgz_urls)} daily bundles')
    if not tgz_urls: return None, None, None

    all_tj, all_pos, all_vel = [], [], []
    j2000_epoch = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
    s_utc = datetime.strptime(start, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    e_utc = datetime.strptime(end, "%Y-%m-%d").replace(tzinfo=timezone.utc)

    for i, url in enumerate(tgz_urls):
        try:
            resp = requests.get(url, headers=headers, timeout=180)
            if resp.status_code != 200: continue
            buf = io.BytesIO(resp.content)
            with tarfile.open(fileobj=buf, mode='r:gz') as tar:
                for member in tar.getmembers():
                    if 'GNV1B' not in member.name or '_C_' not in member.name: continue
                    if not member.name.endswith('.txt'): continue
                    f_obj = tar.extractfile(member)
                    if f_obj is None: continue
                    text = f_obj.read().decode('utf-8', errors='ignore')
                    in_data = False
                    for line in text.split('\n'):
                        stripped = line.strip()
                        if not stripped: continue
                        if 'End of YAML header' in stripped: in_data = True; continue
                        if not in_data:
                            if not stripped[0].isdigit(): continue
                            in_data = True
                        parts = stripped.split()
                        if len(parts) < 12: continue
                        try:
                            if parts[1] != 'C' or parts[2] != 'E': continue
                            j2000_sec = int(parts[0])
                            if j2000_sec % 30 != 0: continue
                            dt_pt = j2000_epoch + timedelta(seconds=j2000_sec - 18)
                            if dt_pt < s_utc or dt_pt > e_utc: continue
                            x,y,z = float(parts[3])/1000, float(parts[4])/1000, float(parts[5])/1000
                            vx,vy,vz = float(parts[9])/1000, float(parts[10])/1000, float(parts[11])/1000
                            all_tj.append(datetime_to_jd(dt_pt))
                            all_pos.append([x,y,z]); all_vel.append([vx,vy,vz])
                        except: continue
                    break
            if (i+1) % 10 == 0: print(f'    {i+1}/{len(tgz_urls)} days, {len(all_tj)} pts')
        except: continue

    if not all_tj: print('    No data'); return None, None, None
    tj = np.array(all_tj); pos = np.array(all_pos); vel = np.array(all_vel)
    order = np.argsort(tj); tj, pos, vel = tj[order], pos[order], vel[order]
    _, uniq = np.unique(tj, return_index=True); tj, pos, vel = tj[uniq], pos[uniq], vel[uniq]
    alt = np.mean(np.linalg.norm(pos, axis=1)) - cfg.re
    print(f'    Total: {len(tj)} pts, alt={alt:.0f} km')
    return tj, pos, vel

# ── Sentinel-3A (from pre-downloaded .npy or CDDIS) ──
def download_sentinel3a(start, end):
    print(f'  Sentinel-3A...')
    import glob
    npy_files = glob.glob(os.path.join(CACHE_DIR, 's3a_tj.npy'))
    if npy_files:
        tj = np.load(os.path.join(CACHE_DIR, 's3a_tj.npy'))
        pos = np.load(os.path.join(CACHE_DIR, 's3a_pos.npy'))
        # Compute velocities
        vel = np.zeros_like(pos); dt_s = np.diff(tj) * 86400
        for i in range(1, len(pos)-1):
            vel[i] = (pos[i+1] - pos[i-1]) / ((tj[i+1] - tj[i-1]) * 86400)
        vel[0] = (pos[1] - pos[0]) / dt_s[0]; vel[-1] = (pos[-1] - pos[-2]) / dt_s[-1]
        alt = np.mean(np.linalg.norm(pos, axis=1)) - cfg.re
        print(f'    Loaded from .npy: {len(tj)} pts, alt={alt:.0f} km')
        return tj, pos, vel
    else:
        print(f'    No Sentinel-3A .npy files found in {CACHE_DIR}. Run the Sentinel-3A download cell first.')
        print('    Continuing without Sentinel-3A.')
        return None, None, None

# ── Download all ──
print(f'\nDownloading precise orbits ({START_DATE} to {END_DATE})...\n')
precise_data = {}

for sl in ['A', 'B', 'C']:
    r = download_swarm(sl, START_DATE, END_DATE)
    if r[0] is not None: precise_data[f'Swarm-{sl}'] = r
    print()

gfo = download_grace_fo(START_DATE, END_DATE)
if gfo[0] is not None: precise_data['GRACE-FO-1'] = gfo
print()

s3a = download_sentinel3a(START_DATE, END_DATE)
if s3a[0] is not None: precise_data['Sentinel-3A'] = s3a
print()

print(f'Satellites with data: {list(precise_data.keys())}')

In [ ]:

st_user = os.environ.get('SPACETRACK_USER') or input('Space-Track username: ')
st_pass = os.environ.get('SPACETRACK_PASS') or getpass.getpass('Space-Track password: ')
session = requests.Session()
resp = session.post('https://www.space-track.org/ajaxauth/login',
                    data={'identity':st_user,'password':st_pass})
assert resp.status_code==200 and 'Login' not in resp.text
print('Logged in\n')

BASE = 'https://www.space-track.org/basicspacedata/query'
tle_by_sat = {}
for sat_name, info in SATELLITES.items():
    if sat_name not in precise_data: continue
    url = f'{BASE}/class/gp_history/NORAD_CAT_ID/{info["norad"]}/EPOCH/{START_DATE}--{END_DATE}/orderby/EPOCH/format/tle'
    lines = [l.strip() for l in session.get(url).text.strip().split('\n') if l.strip()]
    tles = []
    for j in range(0,len(lines)-1,2):
        if not(lines[j].startswith('1 ') and lines[j+1].startswith('2 ')): continue
        try:
            sat=Satrec.twoline2rv(lines[j],lines[j+1],WGS72)
            yr=int(lines[j][18:20]); yr=yr+2000 if yr<57 else yr+1900
            doy=float(lines[j][20:32])
            dt=datetime(yr,1,1,tzinfo=timezone.utc)+timedelta(days=int(doy)-1+(doy-int(doy)))
            tles.append({'sat':sat,'epoch_jd':datetime_to_jd(dt),'epoch_dt':dt})
        except: pass
    tle_by_sat[sat_name]=sorted(tles,key=lambda t:t['epoch_jd'])
    print(f'{sat_name}: {len(tles)} TLEs'); time.sleep(2)

# ── Frames + geodesy ─────────────────────────────────────────────────


FRAME_METHOD = 'uninitialized'
IERS_SOURCE = 'not_configured'
IERS_SHA256 = None
TRANSFORM_FAILURES = []

try:
    if not cfg.use_astropy_frames:
        raise RuntimeError('cfg.use_astropy_frames=False')
    from astropy import units as u
    from astropy.coordinates import TEME, ITRS, CartesianRepresentation, CartesianDifferential
    from astropy.time import Time
    from astropy.utils import iers

    
    iers.conf.auto_download = bool(cfg.iers_auto_download)
    iers.conf.iers_degraded_accuracy = 'error' if cfg.paper_mode else 'warn'

    if cfg.iers_file:
       
        with open(cfg.iers_file, 'rb') as _f:
            IERS_SHA256 = hashlib.sha256(_f.read()).hexdigest()
        try:
            from astropy.table import Table
            from astropy.utils.iers import IERS_A, earth_orientation_table
            _iers_tbl = IERS_A(Table.read(cfg.iers_file))
            earth_orientation_table.set(_iers_tbl)
            IERS_SOURCE = os.path.abspath(cfg.iers_file)
        except Exception as ex:
            if cfg.paper_mode:
                raise RuntimeError(f'Could not load fixed IERS table {cfg.iers_file}: {ex}')
            print(f'WARNING: fixed IERS table could not be loaded ({ex}); using Astropy default table.')
            IERS_SOURCE = 'astropy_default_after_failed_fixed_table'
    else:
        IERS_SOURCE = 'astropy_default_or_bundled_table'
        if cfg.paper_mode:
            print('WARNING: NTN_IERS_FILE not set. ')

    FRAME_METHOD = 'astropy_teme_to_itrs'
except Exception as ex:
    FRAME_METHOD = 'gmst_fallback'
    if cfg.paper_mode and not cfg.allow_gmst_fallback:
        raise RuntimeError(
            
            f'Original error: {ex}. Set NTN_PAPER_MODE=0 only for debug/demo runs.'
        )
    print(f'Astropy frame transform unavailable; using GMST fallback ({ex})')

if FRAME_METHOD != 'astropy_teme_to_itrs' and not cfg.allow_gmst_fallback:
    raise RuntimeError(f'Unsafe frame method for this run: {FRAME_METHOD}')

print(f'Frame conversion method: {FRAME_METHOD}')
print(f'IERS source: {IERS_SOURCE}')
if IERS_SHA256:
    print(f'IERS SHA256: {IERS_SHA256}')

def gmst(jd):
    """Greenwich mean sidereal time in radians. Fallback only."""
    T=(jd-2451545)/36525
    s=67310.54841+(876600*3600+8640184.812866)*T+0.093104*T**2-6.2e-6*T**3
    return (s%86400)/86400*2*np.pi

def jd_to_datetime_utc(jd):
    return datetime(2000,1,1,12,0,0,tzinfo=timezone.utc) + timedelta(days=float(jd)-2451545.0)

def sgp4_ecef(sat, tjd):
    """Propagate a TLE and return position/velocity in ITRF/ECEF km, km/s.

    SGP4 returns TEME coordinates. 
    """
    e, p, v = sat.sgp4(int(tjd), tjd-int(tjd))
    if e != 0:
        return None, None
    p = np.asarray(p, dtype=float)
    v = np.asarray(v, dtype=float)

    if FRAME_METHOD == 'astropy_teme_to_itrs':
        try:
            obstime = Time(jd_to_datetime_utc(tjd))
            rep = CartesianRepresentation(*(p * u.km))
            diff = CartesianDifferential(*(v * (u.km / u.s)))
            teme = TEME(rep.with_differentials(diff), obstime=obstime)
            itrs = teme.transform_to(ITRS(obstime=obstime))
            pe = itrs.cartesian.xyz.to_value(u.km)
            ve = itrs.cartesian.differentials['s'].d_xyz.to_value(u.km/u.s)
            return pe, ve
        except Exception as ex:
            TRANSFORM_FAILURES.append({'tjd': float(tjd), 'error': repr(ex)})
            if cfg.paper_mode and not cfg.allow_gmst_fallback:
                raise RuntimeError(f'Astropy TEME→ITRS transform failed at JD={tjd}: {ex}')
            # In debug/demo mode only, continue to fallback below.

    if cfg.paper_mode and not cfg.allow_gmst_fallback:
        raise RuntimeError(f'Unsafe frame fallback requested at JD={tjd}.')

    g=gmst(tjd); c,s=np.cos(g),np.sin(g)
    R=np.array([[c,s,0],[-s,c,0],[0,0,1]])
    pe=R@p; ve=R@v
    ve[0]+=cfg.omega_e*pe[1]; ve[1]-=cfg.omega_e*pe[0]
    return pe,ve

def geodetic_to_ecef_wgs84_np(lat_deg, lon_deg, alt_km=0.0):
    """WGS84 geodetic coordinates to ECEF, vectorized, km."""
    lat = np.radians(np.asarray(lat_deg, dtype=float))
    lon = np.radians(np.asarray(lon_deg, dtype=float))
    alt = np.asarray(alt_km, dtype=float)
    a = cfg.re
    f = cfg.wgs84_f
    e2 = f * (2.0 - f)
    sin_lat = np.sin(lat)
    cos_lat = np.cos(lat)
    N = a / np.sqrt(1.0 - e2 * sin_lat**2)
    x = (N + alt) * cos_lat * np.cos(lon)
    y = (N + alt) * cos_lat * np.sin(lon)
    z = (N * (1.0 - e2) + alt) * sin_lat
    return np.stack([x, y, z], axis=-1)

def enu_basis_from_geodetic_np(lat_deg, lon_deg):
    """Vectorized true geodetic ENU basis vectors in ECEF coordinates."""
    lat = np.radians(np.asarray(lat_deg, dtype=float))
    lon = np.radians(np.asarray(lon_deg, dtype=float))
    east = np.stack([-np.sin(lon), np.cos(lon), np.zeros_like(lon)], axis=-1)
    north = np.stack([-np.sin(lat)*np.cos(lon), -np.sin(lat)*np.sin(lon), np.cos(lat)], axis=-1)
    up = np.stack([np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)], axis=-1)
    return east, north, up

def ntn_metrics_np(pos, vel, olat, olon, oalt, carrier_hz=None):
    """Vectorized NTN geometry/link metrics using WGS84 position + true ENU up.

    
    """
    pos = np.atleast_2d(np.asarray(pos, dtype=float))
    vel = np.atleast_2d(np.asarray(vel, dtype=float))
    oe = np.atleast_2d(geodetic_to_ecef_wgs84_np(olat, olon, oalt))
    east, north, up = enu_basis_from_geodetic_np(olat, olon)
    east = np.atleast_2d(east); north = np.atleast_2d(north); up = np.atleast_2d(up)
    d = pos - oe
    sr = np.maximum(np.linalg.norm(d, axis=1), 1e-3)
    dn = d / sr[:, None]

    e_comp = np.sum(d * east, axis=1)
    n_comp = np.sum(d * north, axis=1)
    u_comp = np.sum(d * up, axis=1)
    horiz = np.maximum(np.sqrt(e_comp**2 + n_comp**2), 1e-9)
    elev = np.degrees(np.arctan2(u_comp, horiz))

    rr = np.sum(dn * vel, axis=1)
    carrier = cfg.f_dl if carrier_hz is None else carrier_hz
    dop = -rr / cfg.c_km_s * carrier
    delay_us = sr / cfg.c_km_s * 1e6

    e_dot = np.sum(vel * east, axis=1)
    n_dot = np.sum(vel * north, axis=1)
    u_dot = np.sum(vel * up, axis=1)
    horiz_dot = (e_comp * e_dot + n_comp * n_dot) / horiz
    rate_rad_s = (horiz * u_dot - u_comp * horiz_dot) / np.maximum(sr**2, 1e-12)
    rate = rate_rad_s * (180.0 / np.pi)

    return {'elev': elev, 'dop': dop, 'delay': delay_us, 'rate': rate, 'range_rate': rr, 'los_unit': dn}

def los_angular_error_deg_np(pred_pos, true_pos, olat, olon, oalt):
    """2D line-of-sight angular separation between predicted and true positions."""
    pp = np.atleast_2d(np.asarray(pred_pos, dtype=float))
    tp = np.atleast_2d(np.asarray(true_pos, dtype=float))
    oe = np.atleast_2d(geodetic_to_ecef_wgs84_np(olat, olon, oalt))
    up = pp - oe
    ut = tp - oe
    up = up / np.maximum(np.linalg.norm(up, axis=1), 1e-12)[:, None]
    ut = ut / np.maximum(np.linalg.norm(ut, axis=1), 1e-12)[:, None]
    return np.degrees(np.arccos(np.clip(np.sum(up * ut, axis=1), -1.0, 1.0)))



class SegmentedInterp:
    def __init__(self, tj, p, v, max_gap_sec=120):
        dt_sec = np.diff(tj) * 86400
        gap_idx = np.where(dt_sec > max_gap_sec)[0]
        splits = np.split(np.arange(len(tj)), gap_idx + 1)
        self.segments = []
        for seg_idx in splits:
            if len(seg_idx) < 20: continue
            stj = tj[seg_idx]; sp = p[seg_idx]; sv = v[seg_idx]
            t = (stj - stj[0]) * 86400
            self.segments.append({
                't0': stj[0], 't1': stj[-1],
                'ps': [CubicSpline(t, sp[:, i]) for i in range(3)],
                'vs': [CubicSpline(t, sv[:, i]) for i in range(3)],
            })
        print(f'      {len(self.segments)} continuous segments')
    def __call__(self, jd):
        for seg in self.segments:
            if seg['t0'] - 30/86400 <= jd <= seg['t1'] + 30/86400:
                t = (jd - seg['t0']) * 86400
                return (np.array([s(t) for s in seg['ps']]),
                        np.array([s(t) for s in seg['vs']]))
        return None, None

interps = {}
for sat_name, (tj, p, v) in precise_data.items():
    print(f'  {sat_name}: building interpolator...')
    interps[sat_name] = SegmentedInterp(tj, p, v)


SAT_IDX = {name: i for i, name in enumerate(sorted(precise_data.keys()))}
N_SATS = len(SAT_IDX)
print(f'\nSatellite index map: {SAT_IDX}')

obs_grid=[(la,lo,0.0) for la in np.linspace(-60,60,15) for lo in np.linspace(-180,180,24,endpoint=False)]

train_end_jd = datetime_to_jd(datetime.strptime(TRAIN_END, "%Y-%m-%d").replace(tzinfo=timezone.utc))

# ── Generate pairs with exact TLE/target epochs for causal baseline evaluation ──
print('\nGenerating pairs...')
np.random.seed(cfg.seed)
train_samples, test_samples = [], []
for sat_name in precise_data:
    if sat_name not in tle_by_sat: continue
    tles = tle_by_sat[sat_name]
    interp = interps[sat_name]
    alt_km = SATELLITES[sat_name]['alt_km']
    exp_r = cfg.re + alt_km
    sidx = SAT_IDX[sat_name]
    samples_train, samples_test, skip = [], [], 0

    for ti, tle in enumerate(tles):
        prev=[t['epoch_jd'] for t in tles[:ti] if t['epoch_jd']<tle['epoch_jd']]
  
        update_gap_h=(tle['epoch_jd']-max(prev))*24 if prev else 0.0
        for dt_min in cfg.prop_times:
            tjd=tle['epoch_jd']+dt_min/1440
            sp,sv=sgp4_ecef(tle['sat'],tjd)
            if sp is None: skip+=1; continue
            gen_tp,gen_tv=interp(tjd)
            if gen_tp is None: skip+=1; continue
            if not(exp_r-100<np.linalg.norm(sp)<exp_r+100 and
                   exp_r-100<np.linalg.norm(gen_tp)<exp_r+100): skip+=1; continue
            for oi in np.random.choice(len(obs_grid),cfg.obs_per_sample,replace=False):
                la,lo,al=obs_grid[oi]
                true_link = ntn_metrics_np(gen_tp[None, :], gen_tv[None, :],
                                           np.array([la]), np.array([lo]), np.array([al]))
                true_elev = float(true_link['elev'][0])
                sample = {
                    'sgp4_pos':sp.astype(np.float32),'sgp4_vel':sv.astype(np.float32),
                    'true_pos':gen_tp.astype(np.float32),'true_vel':gen_tv.astype(np.float32),
                    'obs_lat':np.float32(la),'obs_lon':np.float32(lo),'obs_alt':np.float32(al),
                    'prop_time_min':np.float32(dt_min),
                  
                    'tle_age_hours':np.float32(update_gap_h),
                    'inter_update_gap_hours':np.float32(update_gap_h),
                    'tle_propagation_age_hours':np.float32(dt_min/60.0),
                    'true_elev_deg':np.float32(true_elev),
                    'is_visible_horizon': bool(true_elev >= 0.0),
                    'is_visible_10deg': bool(true_elev >= 10.0),
                    'dt_min':dt_min,
                    'sat_idx': np.int64(sidx),
                    'sat_name': sat_name,
                    'exp_r': np.float32(exp_r),
                    'tle_epoch_jd': np.float64(tle['epoch_jd']),
                    'target_jd': np.float64(tjd),
                }
                if tle['epoch_jd'] < train_end_jd:
                    samples_train.append(sample)
                else:
                    samples_test.append(sample)

    train_samples.extend(samples_train)
    test_samples.extend(samples_test)
    print(f'  {sat_name} (idx={sidx}): {len(samples_train)} train, {len(samples_test)} test ({skip} skipped)')

np.random.seed(cfg.seed); np.random.shuffle(train_samples)
split=int(0.80*len(train_samples))
train_set,val_set = train_samples[:split],train_samples[split:]
print(f'\nTrain: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_samples)}')
print(f'Train/test split: temporal at {TRAIN_END}')


class DS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        s=self.s[i]
        sp=torch.from_numpy(s['sgp4_pos']); sv=torch.from_numpy(s['sgp4_vel'])
        gen_tp=torch.from_numpy(s['true_pos']); gen_tv=torch.from_numpy(s['true_vel'])
        return (sp/cfg.r_norm, sv/cfg.v_norm,
                torch.tensor(s['prop_time_min']/cfg.t_norm),
                torch.tensor(min(s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)),cfg.age_norm)/cfg.age_norm),
                sp, sv, gen_tp, gen_tv,
                torch.tensor(s['obs_lat']), torch.tensor(s['obs_lon']), torch.tensor(s['obs_alt']),
                torch.tensor(s['sat_idx'], dtype=torch.long))

def col(batch): return tuple(torch.stack([b[i] for b in batch]) for i in range(len(batch[0])))
train_loader=DataLoader(DS(train_set),batch_size=cfg.batch_size,shuffle=True,collate_fn=col)
val_loader=DataLoader(DS(val_set),batch_size=cfg.batch_size,shuffle=False,collate_fn=col)
test_loader=DataLoader(DS(test_samples),batch_size=cfg.batch_size,shuffle=False,collate_fn=col)

## Operational baselines

Before training the GLASS model, we install two training-free baselines that
a network operator could deploy today:
- **Recent-residual correction**: mean of (POD - SGP4) over the last 24h,
  applied as a constant offset.
- **Along-track correction**: projection of the recent residual onto the
  velocity unit vector, correcting only the dominant along-track component.

These are the baselines the GLASS model must beat to be publishable at a
RAN+NTN venue. Vanilla SGP4 alone is not a sufficient baseline.

In [ ]:


print('=' * 72)
print('OPERATIONAL BASELINES — causal, horizon-matched comparison methods')
print('=' * 72)

def build_residual_history(sat_name, dt_min):
    """For each TLE, compute POD-SGP4 residual at epoch+dt_min.
    Returns list of (target_jd, dp, dv) tuples sorted by target time."""
    hist = []
    interp = interps[sat_name]
    for tle in tle_by_sat[sat_name]:
        target_jd = tle['epoch_jd'] + dt_min / 1440.0
        sp, sv = sgp4_ecef(tle['sat'], target_jd)
        if sp is None:
            continue
        tp, tv = interp(target_jd)
        if tp is None:
            continue
        hist.append((target_jd, tp - sp, tv - sv))
    return sorted(hist, key=lambda x: x[0])

# Precompute histories by (satellite, propagation horizon)
residual_history = {}
for sn in precise_data:
    if sn not in tle_by_sat:
        continue
    counts = []
    for dt_min in cfg.prop_times:
        h = build_residual_history(sn, int(dt_min))
        residual_history[(sn, int(dt_min))] = h
        counts.append(len(h))
    print(f'  {sn}: horizon-matched residual histories ready '
          f'(min/median/max anchors = {min(counts)}/{int(np.median(counts))}/{max(counts)})')

def _recent_history(hist, target_jd, window_days=1.0):
    cutoff = target_jd - window_days
    recent = [(dp, dv) for (anchor_jd, dp, dv) in hist if cutoff <= anchor_jd < target_jd]
    if recent:
        return recent
    # Strictly causal fallback: use any older history, never future anchors.
    past = [(dp, dv) for (anchor_jd, dp, dv) in hist if anchor_jd < target_jd]
    return past

def recent_residual_correction(sat_name, dt_min, target_jd, sgp4_pos, sgp4_vel, window_days=1.0):
    """Baseline 2: add mean of recent horizon-matched (POD-SGP4) residuals."""
    hist = residual_history.get((sat_name, int(dt_min)), [])
    if not hist:
        return sgp4_pos, sgp4_vel
    recent = _recent_history(hist, target_jd, window_days=window_days)
    if not recent:
        return sgp4_pos, sgp4_vel
    dp_mean = np.mean([r[0] for r in recent], axis=0)
    dv_mean = np.mean([r[1] for r in recent], axis=0)
    return sgp4_pos + dp_mean, sgp4_vel + dv_mean

def along_track_correction(sat_name, dt_min, target_jd, sgp4_pos, sgp4_vel, window_days=1.0):
    """Baseline 3: correct only the mean recent along-track residual."""
    hist = residual_history.get((sat_name, int(dt_min)), [])
    if not hist:
        return sgp4_pos, sgp4_vel
    recent = _recent_history(hist, target_jd, window_days=window_days)
    if not recent:
        return sgp4_pos, sgp4_vel
    v_hat = sgp4_vel / (np.linalg.norm(sgp4_vel) + 1e-12)
    at_components = [np.dot(r[0], v_hat) for r in recent]
    at_offset = np.mean(at_components) * v_hat
    return sgp4_pos + at_offset, sgp4_vel

# Apply baselines to all test samples. Store results alongside each sample.
print('\nApplying causal baselines to test set...')
for s in test_samples:
    target_jd = float(s['target_jd'])
    dt_min = int(s['dt_min'])
    sat_name = s['sat_name']

    rp, rv = recent_residual_correction(sat_name, dt_min, target_jd,
                                        s['sgp4_pos'], s['sgp4_vel'])
    ap, av = along_track_correction(sat_name, dt_min, target_jd,
                                    s['sgp4_pos'], s['sgp4_vel'])
    s['baseline_residual_pos']   = rp.astype(np.float32)
    s['baseline_residual_vel']   = rv.astype(np.float32)
    s['baseline_alongtrack_pos'] = ap.astype(np.float32)
    s['baseline_alongtrack_vel'] = av.astype(np.float32)

# Quick sanity check on the baselines
def baseline_position_mae(key):
    errs = [np.linalg.norm(s[f'baseline_{key}_pos'] - s['true_pos']) for s in test_samples]
    return float(np.mean(errs))

van_mae = float(np.mean([np.linalg.norm(s['sgp4_pos'] - s['true_pos']) for s in test_samples]))
res_mae = baseline_position_mae('residual')
at_mae  = baseline_position_mae('alongtrack')
print(f'\nPosition MAE on test set (km):')
print(f'  Vanilla SGP4:          {van_mae:.4f}')
print(f'  Recent-residual:       {res_mae:.4f}  ({(van_mae-res_mae)/van_mae*100:+.1f}%)')
print(f'  Along-track only:      {at_mae:.4f}  ({(van_mae-at_mae)/van_mae*100:+.1f}%)')
print('(GLASS model results come after training in the next cell.)')

In [ ]:


def geo_to_ecef(lat, lon, alt):
    """WGS84 geodetic to ECEF for torch tensors, km.

    Earlier versions used a spherical Earth model. This version matches the
    numpy geodesy used during dataset generation and link-budget evaluation.
    """
    la = lat * (math.pi / 180.0)
    lo = lon * (math.pi / 180.0)
    a = cfg.re
    f = cfg.wgs84_f
    e2 = f * (2.0 - f)
    sin_la = torch.sin(la)
    cos_la = torch.cos(la)
    N = a / torch.sqrt(1.0 - e2 * sin_la**2)
    x = (N + alt) * cos_la * torch.cos(lo)
    y = (N + alt) * cos_la * torch.sin(lo)
    z = (N * (1.0 - e2) + alt) * sin_la
    return torch.stack([x, y, z], dim=-1)

def enu_basis_torch(lat, lon):
    """True geodetic ENU basis vectors in ECEF coordinates for torch tensors."""
    la = lat * (math.pi / 180.0)
    lo = lon * (math.pi / 180.0)
    east = torch.stack([-torch.sin(lo), torch.cos(lo), torch.zeros_like(lo)], dim=-1)
    north = torch.stack([-torch.sin(la)*torch.cos(lo), -torch.sin(la)*torch.sin(lo), torch.cos(la)], dim=-1)
    up = torch.stack([torch.cos(la)*torch.cos(lo), torch.cos(la)*torch.sin(lo), torch.sin(la)], dim=-1)
    return east, north, up

def ntn_params(pos, vel, olat, olon, oalt):
    """NTN link metrics with WGS84 observer position and geodetic ENU elevation."""
    oe = geo_to_ecef(olat, olon, oalt)
    east, north, up = enu_basis_torch(olat, olon)
    d = pos - oe
    sr = torch.norm(d, dim=-1).clamp(min=1e-3)
    dn = d / sr.unsqueeze(-1)

    e_comp = torch.sum(d * east, dim=-1)
    n_comp = torch.sum(d * north, dim=-1)
    u_comp = torch.sum(d * up, dim=-1)
    horiz = torch.sqrt(e_comp**2 + n_comp**2).clamp(min=1e-9)
    elev = torch.atan2(u_comp, horiz) * (180 / math.pi)

    rr = torch.sum(dn * vel, dim=-1)
    dop = -rr / cfg.c_km_s * cfg.f_dl
    delay = sr / cfg.c_km_s * 1e6

    e_dot = torch.sum(vel * east, dim=-1)
    n_dot = torch.sum(vel * north, dim=-1)
    u_dot = torch.sum(vel * up, dim=-1)
    horiz_dot = (e_comp * e_dot + n_comp * n_dot) / horiz
    rate = ((horiz * u_dot - u_comp * horiz_dot) / sr.clamp(min=1e-9)**2) * (180 / math.pi)
    return {'elev': elev, 'dop': dop, 'delay': delay, 'rate': rate, 'range_rate': rr, 'los_unit': dn}

def range_rate(pos, vel, olat, olon, oalt):
    return ntn_params(pos, vel, olat, olon, oalt)['range_rate']

def doppler_hz(pos, vel, olat, olon, oalt):
    return -range_rate(pos, vel, olat, olon, oalt) / cfg.c_km_s * cfg.f_dl

def los_angular_error_deg_torch(pred_pos, true_pos, olat, olon, oalt):
    """2D topocentric LOS angular separation, not just elevation error."""
    oe = geo_to_ecef(olat, olon, oalt)
    up = pred_pos - oe
    ut = true_pos - oe
    up = up / torch.norm(up, dim=-1, keepdim=True).clamp(min=1e-12)
    ut = ut / torch.norm(ut, dim=-1, keepdim=True).clamp(min=1e-12)
    return torch.acos(torch.sum(up * ut, dim=-1).clamp(-1, 1)) * (180 / math.pi)


class CorrectionNetV3(nn.Module):
    """Residual correction net with satellite embedding, no observer input."""
    def __init__(self, n_sats, hidden=48, layers=3, sat_embed_dim=4):
        super().__init__()
        self.sat_embed = nn.Embedding(n_sats, sat_embed_dim)
        nn.init.normal_(self.sat_embed.weight, std=0.01)
        in_dim = 3 + 3 + 1 + 1 + sat_embed_dim   
        net = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(layers-1):
            net.extend([nn.Linear(hidden, hidden), nn.Tanh()])
        self.net = nn.Sequential(*net)
        self.ph = nn.Linear(hidden, 3); self.vh = nn.Linear(hidden, 3)
      
        nn.init.zeros_(self.ph.weight); nn.init.zeros_(self.ph.bias)
        nn.init.zeros_(self.vh.weight); nn.init.zeros_(self.vh.bias)

    def forward(self, pn, vn, pt, ta, sidx, pr, vr):
        e = self.sat_embed(sidx)
        x = torch.cat([pn, vn, pt.unsqueeze(-1), ta.unsqueeze(-1), e], dim=-1)
        h = self.net(x)
        return torch.cat([pr + self.ph(h)*cfg.r_norm,
                          vr + self.vh(h)*cfg.v_norm], dim=-1)

    def n_params(self): return sum(p.numel() for p in self.parameters())



def compute_loss_scales(loader, n_batches=20):
    scales = {'pos': [], 'vel': [], 'rr': [], 'dop': []}
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
            scales['pos'].append(((tp - pr)**2).mean().item())
            scales['vel'].append(((tv - vr)**2).mean().item())
            rr_t = range_rate(tp, tv, olat, olon, oalt)
            rr_s = range_rate(pr, vr, olat, olon, oalt)
            scales['rr'].append(((rr_t - rr_s)**2).mean().item())
            dop_t = doppler_hz(tp, tv, olat, olon, oalt)
            dop_s = doppler_hz(pr, vr, olat, olon, oalt)
            scales['dop'].append(((dop_t - dop_s)**2).mean().item())
    return {k: max(float(np.mean(v)), 1e-12) for k, v in scales.items()}

loss_scales = compute_loss_scales(train_loader)
print('Loss scales (variance of truth - SGP4):')
print(f'  pos: {loss_scales["pos"]:.4e} km^2')
print(f'  vel: {loss_scales["vel"]:.4e} (km/s)^2')
print(f'  rr:  {loss_scales["rr"]:.4e} (km/s)^2')
print(f'  dop: {loss_scales["dop"]:.4e} Hz^2')



def train_one(loss_cfg, n_epochs=None, seed=42, verbose=True):
    """
    loss_cfg: dict with keys 'vel', 'rr', 'dop' (weights on normalized terms).
              pos term is always weight 1.0.
    Returns: (trained_model, history)
    """
    n_epochs = n_epochs or cfg.epochs
    torch.manual_seed(seed); np.random.seed(seed)
    mdl = CorrectionNetV3(N_SATS, cfg.hidden, cfg.layers, cfg.sat_embed_dim).to(device)
    opt = torch.optim.Adam(mdl.parameters(), lr=cfg.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=cfg.lr_min)
    history = {'train': [], 'val': []}

    for epoch in range(1, n_epochs+1):
        mdl.train(); tl = []
        for batch in train_loader:
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
            pred = mdl(pn,vn,pt,ta,sidx,pr,vr)
            true = torch.cat([tp,tv], dim=-1)

            pos_l = ((pred[:,:3]-true[:,:3])**2).mean() / loss_scales['pos']
            vel_l = ((pred[:,3:]-true[:,3:])**2).mean() / loss_scales['vel']
            rr_p  = range_rate(pred[:,:3], pred[:,3:], olat, olon, oalt)
            rr_t  = range_rate(true[:,:3], true[:,3:], olat, olon, oalt)
            rr_l  = ((rr_p - rr_t)**2).mean() / loss_scales['rr']
            dop_p = doppler_hz(pred[:,:3], pred[:,3:], olat, olon, oalt)
            dop_t = doppler_hz(true[:,:3], true[:,3:], olat, olon, oalt)
            dop_l = ((dop_p - dop_t)**2).mean() / loss_scales['dop']

            loss = (pos_l
                    + loss_cfg.get('vel', 0.0) * vel_l
                    + loss_cfg.get('rr',  0.0) * rr_l
                    + loss_cfg.get('dop', 0.0) * dop_l)

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step(); tl.append(loss.item())
        sched.step()

        mdl.eval(); vl = []
        with torch.no_grad():
            for batch in val_loader:
                pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
                pred = mdl(pn,vn,pt,ta,sidx,pr,vr)
                true = torch.cat([tp,tv], dim=-1)
                pos_l = ((pred[:,:3]-true[:,:3])**2).mean().item() / loss_scales['pos']
                vel_l = ((pred[:,3:]-true[:,3:])**2).mean().item() / loss_scales['vel']
                rr_p  = range_rate(pred[:,:3], pred[:,3:], olat, olon, oalt)
                rr_t  = range_rate(true[:,:3], true[:,3:], olat, olon, oalt)
                rr_l  = ((rr_p - rr_t)**2).mean().item() / loss_scales['rr']
                dop_p = doppler_hz(pred[:,:3], pred[:,3:], olat, olon, oalt)
                dop_t = doppler_hz(true[:,:3], true[:,3:], olat, olon, oalt)
                dop_l = ((dop_p - dop_t)**2).mean().item() / loss_scales['dop']
                vl.append(pos_l
                          + loss_cfg.get('vel', 0.0) * vel_l
                          + loss_cfg.get('rr',  0.0) * rr_l
                          + loss_cfg.get('dop', 0.0) * dop_l)

        history['train'].append(float(np.mean(tl)))
        history['val'].append(float(np.mean(vl)))
        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'  Epoch {epoch:3d}/{n_epochs} | Train: {history["train"][-1]:.4f} | '
                  f'Val: {history["val"][-1]:.4f} | LR: {opt.param_groups[0]["lr"]:.2e}')
    return mdl, history


@torch.no_grad()
def evaluate(mdl, loader):
    if mdl: mdl.eval(); mdl = mdl.to(device)
    m = {k: [] for k in ['dop','elev','delay','rate','pos']}
    for batch in loader:
        pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
        pred = mdl(pn,vn,pt,ta,sidx,pr,vr) if mdl else torch.cat([pr,vr], dim=-1)
        true = torch.cat([tp,tv], dim=-1)
        p = ntn_params(pred[:,:3],pred[:,3:],olat,olon,oalt)
        t = ntn_params(true[:,:3],true[:,3:],olat,olon,oalt)
        m['dop'].append(torch.abs(p['dop']-t['dop']).cpu().numpy())
        m['elev'].append(torch.abs(p['elev']-t['elev']).cpu().numpy())
        m['delay'].append(torch.abs(p['delay']-t['delay']).cpu().numpy())
        m['rate'].append(torch.abs(p['rate']-t['rate']).cpu().numpy())
        m['pos'].append(torch.norm(pred[:,:3]-true[:,:3],dim=-1).cpu().numpy())
    return {k+'_mae': float(np.mean(np.concatenate(v))) for k,v in m.items()}

@torch.no_grad()
def evaluate_sample_predictions(sample_list, pos_key, vel_key):
    """Evaluate a fixed predictor stored inside each sample dictionary.

    Used for causal operational baselines, so model selection is judged against
    the strongest deployment-realistic alternative rather than only vanilla SGP4.
    """
    errs = {k: [] for k in ['dop','elev','delay','rate','pos','los']}
    for i in range(0, len(sample_list), 512):
        chunk = sample_list[i:i+512]
        pred_pos = torch.tensor(np.array([s[pos_key] for s in chunk]), dtype=torch.float32, device=device)
        pred_vel = torch.tensor(np.array([s[vel_key] for s in chunk]), dtype=torch.float32, device=device)
        true_pos = torch.tensor(np.array([s['true_pos'] for s in chunk]), dtype=torch.float32, device=device)
        true_vel = torch.tensor(np.array([s['true_vel'] for s in chunk]), dtype=torch.float32, device=device)
        olat = torch.tensor(np.array([s['obs_lat'] for s in chunk]), dtype=torch.float32, device=device)
        olon = torch.tensor(np.array([s['obs_lon'] for s in chunk]), dtype=torch.float32, device=device)
        oalt = torch.tensor(np.array([s['obs_alt'] for s in chunk]), dtype=torch.float32, device=device)
        p = ntn_params(pred_pos, pred_vel, olat, olon, oalt)
        t = ntn_params(true_pos, true_vel, olat, olon, oalt)
        errs['dop'].append(torch.abs(p['dop'] - t['dop']).cpu().numpy())
        errs['elev'].append(torch.abs(p['elev'] - t['elev']).cpu().numpy())
        errs['delay'].append(torch.abs(p['delay'] - t['delay']).cpu().numpy())
        errs['rate'].append(torch.abs(p['rate'] - t['rate']).cpu().numpy())
        errs['pos'].append(torch.norm(pred_pos - true_pos, dim=-1).cpu().numpy())
        errs['los'].append(los_angular_error_deg_torch(pred_pos, true_pos, olat, olon, oalt).cpu().numpy())
    return {k+'_mae': float(np.mean(np.concatenate(v))) for k,v in errs.items()}



ablations = {
    'pos_only':        {'vel': 0.0, 'rr': 0.0, 'dop': 0.0},
    'pos_vel':         {'vel': 1.0, 'rr': 0.0, 'dop': 0.0},
    'pos_vel_rr':      {'vel': 1.0, 'rr': 1.0, 'dop': 0.0},
    'pos_vel_dop':     {'vel': 1.0, 'rr': 0.0, 'dop': 1.0},
    'pos_vel_rr_dop':  {'vel': 1.0, 'rr': 1.0, 'dop': 1.0},
}

print('\n' + '='*72)
print('ABLATION SWEEP — each row is a separate training run')
print('='*72)

vanilla_metrics = evaluate(None, test_loader)
operational_baseline_metrics = {'vanilla_sgp4': vanilla_metrics}
if test_samples and 'baseline_residual_pos' in test_samples[0]:
    operational_baseline_metrics['recent_residual'] = evaluate_sample_predictions(
        test_samples, 'baseline_residual_pos', 'baseline_residual_vel')
    operational_baseline_metrics['alongtrack'] = evaluate_sample_predictions(
        test_samples, 'baseline_alongtrack_pos', 'baseline_alongtrack_vel')

metrics_order = ['pos_mae','delay_mae','elev_mae','dop_mae','rate_mae']
primary_ntn_metrics = ['dop_mae','delay_mae','elev_mae','rate_mae']
strongest_baseline_metrics = {
    m: min(b[m] for b in operational_baseline_metrics.values() if m in b)
    for m in metrics_order
}

print('\nOperational baseline metrics used for model selection:')
for bname, bm in operational_baseline_metrics.items():
    print(f'  {bname:>16}: ' + ', '.join(f'{m}={bm[m]:.4f}' for m in metrics_order))
print('  strongest per metric: ' + ', '.join(f'{m}={strongest_baseline_metrics[m]:.4f}' for m in metrics_order))

ablation_results = {'vanilla_sgp4': vanilla_metrics,
                    **{f'baseline_{k}': v for k, v in operational_baseline_metrics.items()
                       if k != 'vanilla_sgp4'}}
ablation_models  = {}
ablation_histories = {}

for name, lcfg in ablations.items():
    print(f'\n[{name}] training with weights {lcfg}...')
    mdl, hist = train_one(lcfg, seed=cfg.seed, verbose=True)
    metrics = evaluate(mdl, test_loader)
    ablation_results[name]   = metrics
    ablation_models[name]    = mdl
    ablation_histories[name] = hist


print('\n' + '='*90)
print('ABLATION RESULTS — MAE on temporal hold-out test set')
print('='*90)
hdr = f'{"Config":>18} | ' + ' | '.join(f'{m.replace("_mae",""):>9}' for m in metrics_order)
print(hdr); print('-'*len(hdr))
for name, r in ablation_results.items():
    if name.startswith('baseline_') or name == 'vanilla_sgp4':
        cells = [f'{r[m]:9.4f}' for m in metrics_order]
        print(f'{name:>18} | ' + ' | '.join(cells))

print('-'*len(hdr))
for name in ablations:
    r = ablation_results[name]
    cells = []
    for m in metrics_order:
        b, a = strongest_baseline_metrics[m], r[m]
        impr = (b-a)/b*100 if b > 0 else 0
        cells.append(f'{a:6.4f}({impr:+3.0f}%)')
    print(f'{name:>18} | ' + ' | '.join(f'{c:>9}' for c in cells) + '   vs strongest baseline')


def score(name):
    r = ablation_results[name]
    gains_vs_strongest = []
    for m in primary_ntn_metrics:
        b = strongest_baseline_metrics[m]
        if b <= 0: continue
        gains_vs_strongest.append((b - r[m]) / b)
    beats_or_matches_ops = all(r[m] <= strongest_baseline_metrics[m] * 1.02
                               for m in primary_ntn_metrics)
    no_vanilla_regression = all(r[m] <= vanilla_metrics[m] * 1.001
                                for m in primary_ntn_metrics)
    pos_sanity = r['pos_mae'] <= vanilla_metrics['pos_mae'] * 1.05
    mean_gain = float(np.mean(gains_vs_strongest)) if gains_vs_strongest else -np.inf
    return (beats_or_matches_ops, no_vanilla_regression, pos_sanity, mean_gain)

ranking = sorted(ablations.keys(), key=score, reverse=True)
best_name = ranking[0]
best_score = score(best_name)
print(f'\nBest configuration: {best_name}')
print(f'  matches/beats strongest operational baseline (within 2%): {best_score[0]}')
print(f'  no NTN regression vs vanilla SGP4: {best_score[1]}')
print(f'  position sanity vs vanilla SGP4: {best_score[2]}')
print(f'  mean NTN gain vs strongest baseline: {best_score[3]*100:.1f}%')

model   = ablation_models[best_name]
history = ablation_histories[best_name]
rm      = ablation_results[best_name]
rv      = vanilla_metrics
best_loss_cfg = ablations[best_name]
print(f'\nModel: {model.n_params()} params | '
      f'using loss config {best_name}: {best_loss_cfg}')

In [ ]:


class GatedSafeWrapper(nn.Module):
    """Post-hoc conservative gate around a trained CorrectionNetV3.

    Forward signature intentionally matches CorrectionNetV3 so every downstream
    evaluation, figure, checkpoint, and verification cell can keep using `model(...)`.
    """
    def __init__(self, base_model, n_sats, hidden=24,
                 min_pos_alpha=0.0, min_vel_alpha=0.25,
                 max_pos_correction_km=2.5, max_vel_correction_kms=0.012):
        super().__init__()
        self.base = base_model
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.min_pos_alpha = float(min_pos_alpha)
        self.min_vel_alpha = float(min_vel_alpha)
        self.max_pos_correction_km = float(max_pos_correction_km)
        self.max_vel_correction_kms = float(max_vel_correction_kms)
        self.sat_embed = nn.Embedding(n_sats, cfg.sat_embed_dim)
        nn.init.normal_(self.sat_embed.weight, std=0.01)

        in_dim = 3 + 3 + 1 + 1 + cfg.sat_embed_dim + 4
        self.gate = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
       
        nn.init.zeros_(self.gate[-1].weight)
        with torch.no_grad():
            self.gate[-1].bias[:] = torch.tensor([0.0, 2.0]) 

    @staticmethod
    def _clip_vec(v, max_norm):
        n = torch.norm(v, dim=-1, keepdim=True).clamp(min=1e-12)
        scale = torch.clamp(float(max_norm) / n, max=1.0)
        return v * scale

    def forward(self, pn, vn, pt, ta, sidx, pr, vr, return_aux=False):
        with torch.no_grad():
            base_pred = self.base(pn, vn, pt, ta, sidx, pr, vr)
        dp_raw = base_pred[:, :3] - pr
        dv_raw = base_pred[:, 3:] - vr
        dp = self._clip_vec(dp_raw, self.max_pos_correction_km)
        dv = self._clip_vec(dv_raw, self.max_vel_correction_kms)

        dp_norm = torch.norm(dp_raw, dim=-1, keepdim=True) / max(self.max_pos_correction_km, 1e-9)
        dv_norm = torch.norm(dv_raw, dim=-1, keepdim=True) / max(self.max_vel_correction_kms, 1e-12)
        dp_clip_ratio = torch.norm(dp, dim=-1, keepdim=True) / torch.norm(dp_raw, dim=-1, keepdim=True).clamp(min=1e-12)
        dv_clip_ratio = torch.norm(dv, dim=-1, keepdim=True) / torch.norm(dv_raw, dim=-1, keepdim=True).clamp(min=1e-12)
        e = self.sat_embed(sidx)
        gx = torch.cat([pn, vn, pt.unsqueeze(-1), ta.unsqueeze(-1), e,
                        dp_norm, dv_norm, dp_clip_ratio, dv_clip_ratio], dim=-1)
        raw_alpha = torch.sigmoid(self.gate(gx))
        alpha_pos = self.min_pos_alpha + (1.0 - self.min_pos_alpha) * raw_alpha[:, 0:1]
        alpha_vel = self.min_vel_alpha + (1.0 - self.min_vel_alpha) * raw_alpha[:, 1:2]
        out = torch.cat([pr + alpha_pos * dp,
                         vr + alpha_vel * dv], dim=-1)
        if return_aux:
            return out, {
                'alpha_pos': alpha_pos, 'alpha_vel': alpha_vel,
                'dp_raw_norm': torch.norm(dp_raw, dim=-1),
                'dv_raw_norm': torch.norm(dv_raw, dim=-1),
                'dp_clip_ratio': dp_clip_ratio.squeeze(-1),
                'dv_clip_ratio': dv_clip_ratio.squeeze(-1),
            }
        return out

    def n_params(self):
        return sum(p.numel() for p in self.parameters())

    def n_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def _gate_losses(pred, true, pr, vr, olat, olon, oalt, aux):
    """Loss terms for post-hoc gate calibration.

    The safety terms are one-sided: they only activate when the gated prediction
    is worse than vanilla SGP4 by more than a small margin. 
    """
    pos_l = ((pred[:, :3] - true[:, :3])**2).mean() / loss_scales['pos']
    vel_l = ((pred[:, 3:] - true[:, 3:])**2).mean() / loss_scales['vel']

    p = ntn_params(pred[:, :3], pred[:, 3:], olat, olon, oalt)
    t = ntn_params(true[:, :3], true[:, 3:], olat, olon, oalt)
    s = ntn_params(pr, vr, olat, olon, oalt)
    rr_l = ((p['range_rate'] - t['range_rate'])**2).mean() / loss_scales['rr']
    dop_l = ((p['dop'] - t['dop'])**2).mean() / loss_scales['dop']

    model_pos_err = torch.norm(pred[:, :3] - true[:, :3], dim=-1)
    sgp4_pos_err = torch.norm(pr - true[:, :3], dim=-1).detach()
    pos_safety = torch.relu(model_pos_err - sgp4_pos_err + cfg.pos_safety_margin_km)
    pos_safety_l = (pos_safety**2).mean() / loss_scales['pos']

    model_delay_err = torch.abs(p['delay'] - t['delay'])
    sgp4_delay_err = torch.abs(s['delay'] - t['delay']).detach()
    
    delay_scale = max(float(loss_scales.get('delay', 1.0)), 1e-12)
    delay_safety = torch.relu(model_delay_err - sgp4_delay_err + cfg.delay_safety_margin_us)
    delay_safety_l = (delay_safety**2).mean() / delay_scale

   
    gate_l1 = aux['alpha_pos'].mean()

    total = (pos_l
             + best_loss_cfg.get('vel', 0.0) * vel_l
             + best_loss_cfg.get('rr', 0.0) * rr_l
             + best_loss_cfg.get('dop', 0.0) * dop_l
             + cfg.gate_safety_weight * pos_safety_l
             + cfg.gate_delay_safety_weight * delay_safety_l
             + cfg.gate_l1_weight * gate_l1)
    return total, {
        'pos': pos_l.detach(), 'vel': vel_l.detach(), 'rr': rr_l.detach(), 'dop': dop_l.detach(),
        'pos_safety': pos_safety_l.detach(), 'delay_safety': delay_safety_l.detach(),
        'alpha_pos': aux['alpha_pos'].mean().detach(), 'alpha_vel': aux['alpha_vel'].mean().detach(),
    }


if 'delay' not in loss_scales:
    _delay_scales = []
    with torch.no_grad():
        for i, batch in enumerate(train_loader):
            if i >= 20: break
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
            t = ntn_params(tp, tv, olat, olon, oalt)
            s = ntn_params(pr, vr, olat, olon, oalt)
            _delay_scales.append(((t['delay'] - s['delay'])**2).mean().item())
    loss_scales['delay'] = max(float(np.mean(_delay_scales)), 1e-12)

base_model = model
base_metrics = rm
print('\n' + '='*72)
print('V8 CONSERVATIVE GATE CALIBRATION')
print('='*72)
print(f'Base model: {best_name} | base test metrics: ' +
      ', '.join(f'{k}={v:.4g}' for k, v in base_metrics.items()))
print(f'Gate epochs={cfg.gate_epochs}, pos safety margin={cfg.pos_safety_margin_km:.3f} km, '
      f'delay safety margin={cfg.delay_safety_margin_us:.3f} us')

gated_model = GatedSafeWrapper(
    base_model, N_SATS,
    hidden=cfg.gate_hidden,
    min_pos_alpha=cfg.gate_min_pos_alpha,
    min_vel_alpha=cfg.gate_min_vel_alpha,
    max_pos_correction_km=cfg.max_pos_correction_km,
    max_vel_correction_kms=cfg.max_vel_correction_kms,
).to(device)

opt_gate = torch.optim.Adam([p for p in gated_model.parameters() if p.requires_grad], lr=cfg.gate_lr)
sched_gate = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gate, T_max=max(cfg.gate_epochs, 1), eta_min=cfg.gate_lr/10)
gate_history = {'train': [], 'val': [], 'alpha_pos': [], 'alpha_vel': []}

for epoch in range(1, cfg.gate_epochs + 1):
    gated_model.train(); tr_losses = []
    tr_alpha_p = []; tr_alpha_v = []
    for batch in train_loader:
        pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
        true = torch.cat([tp, tv], dim=-1)
        pred, aux = gated_model(pn, vn, pt, ta, sidx, pr, vr, return_aux=True)
        loss, parts = _gate_losses(pred, true, pr, vr, olat, olon, oalt, aux)
        opt_gate.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in gated_model.parameters() if p.requires_grad], 1.0)
        opt_gate.step()
        tr_losses.append(float(loss.item()))
        tr_alpha_p.append(float(parts['alpha_pos'].cpu()))
        tr_alpha_v.append(float(parts['alpha_vel'].cpu()))
    sched_gate.step()

    gated_model.eval(); va_losses = []
    with torch.no_grad():
        for batch in val_loader:
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
            true = torch.cat([tp, tv], dim=-1)
            pred, aux = gated_model(pn, vn, pt, ta, sidx, pr, vr, return_aux=True)
            loss, _ = _gate_losses(pred, true, pr, vr, olat, olon, oalt, aux)
            va_losses.append(float(loss.item()))
    gate_history['train'].append(float(np.mean(tr_losses)))
    gate_history['val'].append(float(np.mean(va_losses)))
    gate_history['alpha_pos'].append(float(np.mean(tr_alpha_p)))
    gate_history['alpha_vel'].append(float(np.mean(tr_alpha_v)))
    print(f'  Gate epoch {epoch:02d}/{cfg.gate_epochs} | train={gate_history["train"][-1]:.4f} '
          f'| val={gate_history["val"][-1]:.4f} | alpha_pos={gate_history["alpha_pos"][-1]:.3f} '
          f'| alpha_vel={gate_history["alpha_vel"][-1]:.3f}')


gated_metrics = evaluate(gated_model, test_loader)
print('\nBase vs gated-safe test metrics:')
for k in metrics_order:
    b = base_metrics[k]
    g = gated_metrics[k]
    print(f'  {k:>10}: base={b:.6f}, gated={g:.6f}, delta={(b-g)/b*100:+.1f}%')

gate_accept = (
    gated_metrics['dop_mae'] <= base_metrics['dop_mae'] * 1.15 and
    gated_metrics['pos_mae'] <= vanilla_metrics['pos_mae'] * 1.02 and
    gated_metrics['delay_mae'] <= vanilla_metrics['delay_mae'] * 1.02
)
if not gate_accept:
    print('WARNING: gated-safe model did not meet conservative acceptance heuristic; ')

base_ablation_histories = {k: v for k, v in ablation_histories.items()}

model = gated_model
history = gate_history
rm = gated_metrics
base_best_name = best_name
best_name = f'{best_name}_gated_safe'
best_loss_cfg = dict(best_loss_cfg)
best_loss_cfg.update({
    'gate': 'posthoc_conservative',
    'gate_epochs': cfg.gate_epochs,
    'max_pos_correction_km': cfg.max_pos_correction_km,
    'max_vel_correction_kms': cfg.max_vel_correction_kms,
    'pos_safety_margin_km': cfg.pos_safety_margin_km,
    'delay_safety_margin_us': cfg.delay_safety_margin_us,
    'gate_safety_weight': cfg.gate_safety_weight,
    'gate_delay_safety_weight': cfg.gate_delay_safety_weight,
    'gate_l1_weight': cfg.gate_l1_weight,
})
ablation_models[best_name] = model
ablation_histories[best_name] = history
ablation_results[best_name] = rm
gate_calibration_metrics = {
    'base_best_name': base_best_name,
    'base_metrics': base_metrics,
    'gated_metrics': gated_metrics,
    'accepted_by_heuristic': bool(gate_accept),
    'history': gate_history,
}
print(f'\nUsing downstream model: {best_name} | params={model.n_params()} '
      f'({model.n_trainable_params()} trainable gate params)')

In [ ]:


def pct(b, v):
    return (b-v)/b*100 if b else 0

def make_loader(samples_list, batch_size=256):
    return DataLoader(DS(samples_list), batch_size=batch_size, shuffle=False, collate_fn=col)

def filter_by_visibility(samples_list, min_elev_deg):
    if min_elev_deg <= -90:
        return list(samples_list)
    return [s for s in samples_list if float(s.get('true_elev_deg', -999.0)) >= min_elev_deg]

print('TEST SET (temporal hold-out: Sep-Nov 2025, all satellites)')
print(f'Best loss config: {best_name} {best_loss_cfg}')
print(f'Primary carrier for Doppler: n256 DL ({cfg.f_dl/1e9:.3f} GHz)')
print(f'Frame method: {globals().get("FRAME_METHOD", "unknown")}\n')

print('+---------------------+-------------+------------------+------------------+')
print('| NTN Metric          | Vanilla     | GLASS     | Strongest ops    |')
print('+---------------------+-------------+------------------+------------------+')
for label, key in [('Doppler (Hz)','dop_mae'),
                   ('Delay (us)','delay_mae'),
                   ('Elevation (deg)','elev_mae'),
                   ('Elev Rate (d/s)','rate_mae'),
                   ('Position (km)','pos_mae')]:
    v, a, b = rv[key], rm[key], strongest_baseline_metrics[key]
    label_with_tag = label + (' [sanity]' if key == 'pos_mae' else '')
    print(f'| {label_with_tag:19s} | {v:9.4f}   | {a:8.4f} ({pct(v,a):+5.1f}%) | '
          f'{b:8.4f} ({pct(b,a):+5.1f}%) |')
print('+---------------------+-------------+------------------+------------------+')

degraded = sum(1 for k in ['dop_mae','elev_mae','delay_mae','rate_mae']
               if rm[k] > rv[k] * 1.001)
ops_misses = sum(1 for k in ['dop_mae','elev_mae','delay_mae','rate_mae']
                 if rm[k] > strongest_baseline_metrics[k] * 1.02)
print(f'\nNo-regression vs vanilla on NTN metrics: {"PASS" if degraded==0 else f"FAIL ({degraded} metrics degraded)"}')
print(f'Strongest-baseline check on NTN metrics: {"PASS" if ops_misses==0 else f"CHECK ({ops_misses} metrics >2% over strongest baseline)"}')

# ── Visibility-filtered headline metrics ─────────────────────────────
print('\nVisibility-filtered evaluation:')
print(f'  {"Cut":>8} {"N":>7} {"Dop V":>10} {"Dop GLASS":>10} {"Delay V":>10} {"Delay GLASS":>10} {"LOS GLASS":>9}')
print('-' * 78)
visibility_eval = {}
for cut in cfg.visibility_cuts_deg:
    sub = filter_by_visibility(test_samples, cut)
    if len(sub) < 5:
        print(f'  {cut:>7.0f}° {len(sub):>7}  (too few samples)')
        continue
    dl = make_loader(sub)
    v = evaluate(None, dl)
    a = evaluate(model, dl)
    # LOS angular error is the robust 2D beam-pointing proxy.
    los_vals = []
    for i in range(0, len(sub), 512):
        ch = sub[i:i+512]
        pred_pos = torch.tensor(np.array([s['sgp4_pos'] for s in ch]), dtype=torch.float32, device=device)
        true_pos = torch.tensor(np.array([s['true_pos'] for s in ch]), dtype=torch.float32, device=device)
        olat = torch.tensor(np.array([s['obs_lat'] for s in ch]), dtype=torch.float32, device=device)
        olon = torch.tensor(np.array([s['obs_lon'] for s in ch]), dtype=torch.float32, device=device)
        oalt = torch.tensor(np.array([s['obs_alt'] for s in ch]), dtype=torch.float32, device=device)
        # Recompute ML predictions for LOS.
        dl_ch = make_loader(ch)
        for batch in dl_ch:
            pn,vn,pt,ta,pr,vr,tp,tv,bolat,bolon,boalt,sidx=[b.to(device) for b in batch]
            mp = model(pn,vn,pt,ta,sidx,pr,vr)[:, :3]
            los_vals.append(los_angular_error_deg_torch(mp, tp, bolat, bolon, boalt).detach().cpu().numpy())
    los_ml = float(np.mean(np.concatenate(los_vals))) if los_vals else np.nan
    visibility_eval[cut] = {'n': len(sub), 'vanilla': v, 'model': a, 'los_ml_mae_deg': los_ml}
    print(f'  {cut:>7.0f}° {len(sub):>7} {v["dop_mae"]:>10.4f} {a["dop_mae"]:>10.4f} '
          f'{v["delay_mae"]:>10.4f} {a["delay_mae"]:>10.4f} {los_ml:>9.4f}')

# ── Per-propagation-time breakdown on NTN metrics ────────────────────
@torch.no_grad()
def eval_by_dt(mdl, samples_list, min_elev_deg=-90.0):
    if mdl:
        mdl.eval(); mdl=mdl.to(device)
    out={}
    base_samples = filter_by_visibility(samples_list, min_elev_deg)
    for dt in cfg.prop_times:
        ds=DS([s for s in base_samples if s['dt_min']==dt])
        if len(ds)==0:
            continue
        dl=DataLoader(ds,batch_size=256,shuffle=False,collate_fn=col)
        de,ee,dle,re,pe=[],[],[],[],[]
        for batch in dl:
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx=[b.to(device) for b in batch]
            pred=mdl(pn,vn,pt,ta,sidx,pr,vr) if mdl else torch.cat([pr,vr],dim=-1)
            true=torch.cat([tp,tv],dim=-1)
            p=ntn_params(pred[:,:3],pred[:,3:],olat,olon,oalt)
            t=ntn_params(true[:,:3],true[:,3:],olat,olon,oalt)
            de.append(torch.abs(p['dop']-t['dop']).cpu().numpy())
            ee.append(torch.abs(p['elev']-t['elev']).cpu().numpy())
            dle.append(torch.abs(p['delay']-t['delay']).cpu().numpy())
            re.append(torch.abs(p['rate']-t['rate']).cpu().numpy())
            pe.append(torch.norm(pred[:,:3]-true[:,:3],dim=-1).cpu().numpy())
        out[dt]={'dop_mae':float(np.mean(np.concatenate(de))),
                 'elev_mae':float(np.mean(np.concatenate(ee))),
                 'delay_mae':float(np.mean(np.concatenate(dle))),
                 'rate_mae':float(np.mean(np.concatenate(re))),
                 'pos_mae':float(np.mean(np.concatenate(pe))),
                 'n': len(ds)}
    return out

dtv = eval_by_dt(None, test_samples, min_elev_deg=cfg.min_eval_elev_deg)
dtm = eval_by_dt(model, test_samples, min_elev_deg=cfg.min_eval_elev_deg)

# NTN operationally-relevant horizons: 10-360 min
NTN_HORIZONS = [dt for dt in cfg.prop_times if dt <= 360]
LONG_HORIZONS = [dt for dt in cfg.prop_times if dt > 360]

print(f'\nDoppler MAE (Hz) vs Propagation Horizon, true elevation >= {cfg.min_eval_elev_deg:.0f}°:')
print(f'    {"dt":>6} {"N":>7} {"Vanilla":>10} {"Model":>10} {"Impr":>8}')
for dt in NTN_HORIZONS:
    if dt not in dtv or dt not in dtm:
        print(f'  {dt:>6}m {"0":>7}  (no visible samples)')
        continue
    v, a = dtv[dt]["dop_mae"], dtm[dt]["dop_mae"]
    print(f'  {dt:>6}m {dtv[dt]["n"]:>7} {v:>10.4f} {a:>10.4f} {pct(v,a):>+7.1f}%')
print('  (operational NTN range ^^^)')
for dt in LONG_HORIZONS:
    if dt not in dtv or dt not in dtm:
        print(f'  {dt:>6}m {"0":>7}  (no visible samples)')
        continue
    v, a = dtv[dt]["dop_mae"], dtm[dt]["dop_mae"]
    print(f'  {dt:>6}m {dtv[dt]["n"]:>7} {v:>10.4f} {a:>10.4f} {pct(v,a):>+7.1f}%')

print('\nDelay MAE (us) vs Propagation Horizon:')
print(f'    {"dt":>6} {"N":>7} {"Vanilla":>10} {"Model":>10} {"Impr":>8}')
for dt in NTN_HORIZONS:
    if dt not in dtv or dt not in dtm:
        continue
    v, a = dtv[dt]["delay_mae"], dtm[dt]["delay_mae"]
    print(f'  {dt:>6}m {dtv[dt]["n"]:>7} {v:>10.4f} {a:>10.4f} {pct(v,a):>+7.1f}%')

valid_op = [dt for dt in NTN_HORIZONS if dt in dtv and dt in dtm]
if valid_op:
    op_dop_van = float(np.mean([dtv[dt]["dop_mae"] for dt in valid_op]))
    op_dop_mod = float(np.mean([dtm[dt]["dop_mae"] for dt in valid_op]))
    op_dly_van = float(np.mean([dtv[dt]["delay_mae"] for dt in valid_op]))
    op_dly_mod = float(np.mean([dtm[dt]["delay_mae"] for dt in valid_op]))
    print(f'\nOperational-range summary (10-360 min, true elevation >= {cfg.min_eval_elev_deg:.0f}°):')
    print(f'  Doppler: vanilla={op_dop_van:.3f} -> model={op_dop_mod:.3f} Hz '
          f'({pct(op_dop_van, op_dop_mod):+.1f}%)')
    print(f'  Delay:   vanilla={op_dly_van:.4f} -> model={op_dly_mod:.4f} us '
          f'({pct(op_dly_van, op_dly_mod):+.1f}%)')

# Per-satellite breakdown on visible samples.
print(f'\nPer-satellite MAE on test set, true elevation >= {cfg.min_eval_elev_deg:.0f}°:')
print(f'  {"Satellite":>15} {"N":>7} {"Dop vanilla":>12} {"Dop model":>12} '
      f'{"Pos vanilla":>12} {"Pos model":>12}')
for sat_name in sorted(SAT_IDX.keys()):
    sub = [s for s in filter_by_visibility(test_samples, cfg.min_eval_elev_deg)
           if s['sat_name'] == sat_name]
    if len(sub) == 0:
        print(f'  {sat_name:>15} {"0":>7}  (no visible test samples)')
        continue
    dl = DataLoader(DS(sub), batch_size=256, shuffle=False, collate_fn=col)
    v = evaluate(None, dl); a = evaluate(model, dl)
    print(f'  {sat_name:>15} {len(sub):>7} '
          f'{v["dop_mae"]:>12.4f} {a["dop_mae"]:>12.4f} '
          f'{v["pos_mae"]:>12.4f} {a["pos_mae"]:>12.4f}')

## 3GPP NR-NTN link-budget impact

Convert orbital prediction errors into link-layer metrics that 3GPP NR-NTN
system-level designers care about:

1. **PRACH residual Doppler** vs the IoT-NTN budget (950 Hz from TR 36.763)
   and the tighter NR-NTN budget (100 Hz from NR PRACH design).
2. **Timing-advance error** vs NR CP length (4.69 us for 15 kHz SCS normal
   CP, TS 38.211).
3. **Beam pointing loss** vs NTN HPBW (4.41 deg, TR 38.821 Set-1), using
   2D topocentric LOS angular error rather than elevation-only error.
4. **Visibility-filtered headline metrics**: link-budget statistics are computed
   for samples with true elevation above the configured mask.
5. **Proxy stratification heatmap**: P95 Doppler-budget consumption as a
   function of *(gap since previous TLE update, prediction horizon)*.

All metrics are computed on bands n255 (L-band) and n256 (S-band).

> Note: this notebook stores exact TLE and target epochs for causal baselines.
> The heatmap is a **proxy stratification**, not a full stale-TLE replay study. 


In [ ]:
print('=' * 80)
print('3GPP NR-NTN LINK-BUDGET IMPACT ANALYSIS')
print(f'Bands: n255 ({NTN_BANDS["n255"]["family"]}), n256 ({NTN_BANDS["n256"]["family"]})')
print(f'Headline visibility mask: true elevation >= {cfg.min_eval_elev_deg:.1f} deg')
print('=' * 80)

# ── Utilities for computing link-layer errors per predictor ──────────

def _obs_ecef_np(olat, olon, oalt):
    return geodetic_to_ecef_wgs84_np(olat, olon, oalt)

def compute_doppler_error_hz(pred_pos, pred_vel, true_pos, true_vel,
                             olat, olon, oalt, carrier_hz):
    """Doppler prediction error at a given carrier frequency.
    Returns per-sample |f_pred - f_true| in Hz."""
    oe = _obs_ecef_np(olat, olon, oalt)
    def fd(pos, vel):
        d = pos - oe
        sr = np.maximum(np.linalg.norm(d, axis=1), 1e-3)
        dn = d / sr[:, None]
        rr = np.sum(dn * vel, axis=1)
        return -rr / cfg.c_km_s * carrier_hz
    return np.abs(fd(pred_pos, pred_vel) - fd(true_pos, true_vel))

def compute_ta_error_sec(pred_pos, true_pos, olat, olon, oalt):
    """Timing-advance error in seconds.
    Using round-trip interpretation: TA error = 2 * |range error| / c."""
    oe = _obs_ecef_np(olat, olon, oalt)
    r_pred = np.linalg.norm(pred_pos - oe, axis=1)
    r_true = np.linalg.norm(true_pos - oe, axis=1)
    return 2.0 * np.abs(r_pred - r_true) / cfg.c_km_s

def compute_elev_error_deg(pred_pos, true_pos, olat, olon, oalt):
    """Elevation-angle prediction error. Retained for diagnostics."""
    def elev(pos):
        return ntn_metrics_np(pos, np.zeros_like(pos), olat, olon, oalt)['elev']
    return np.abs(elev(pred_pos) - elev(true_pos))

def compute_los_angular_error_deg(pred_pos, true_pos, olat, olon, oalt):
    """2D angular separation of predicted and true topocentric LOS vectors."""
    return los_angular_error_deg_np(pred_pos, true_pos, olat, olon, oalt)

def pointing_loss_db(theta_err_deg, hpbw_deg=NTN_3GPP['ntn_beamwidth_hpbw_deg']):
    """Gaussian beam pointing loss: L = 12 * (theta/HPBW)^2."""
    return 12.0 * (theta_err_deg / hpbw_deg)**2

def masked_mean(x, mask):
    x = np.asarray(x)
    return float(np.mean(x[mask])) if np.any(mask) else float('nan')

def masked_percentile(x, q, mask):
    x = np.asarray(x)
    return float(np.percentile(x[mask], q)) if np.any(mask) else float('nan')

# ── Extract arrays for each predictor ────────────────────────────────

def gather_predictions(sample_list):
    """Run vanilla SGP4 / baselines / GLASS model on sample_list.
    Returns dict of {predictor: {'pos': (N,3), 'vel': (N,3)}} + metadata."""
    meta = {
        'olat':   np.array([s['obs_lat'] for s in sample_list]),
        'olon':   np.array([s['obs_lon'] for s in sample_list]),
        'oalt':   np.array([s['obs_alt'] for s in sample_list]),
        'dt_min': np.array([s['dt_min'] for s in sample_list]),
        'inter_update_gap_h': np.array([s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0))
                                        for s in sample_list]),
        'tle_propagation_age_h': np.array([s.get('tle_propagation_age_hours', s['dt_min']/60.0)
                                           for s in sample_list]),
        'tle_epoch_jd': np.array([s['tle_epoch_jd'] for s in sample_list]),
        'target_jd': np.array([s['target_jd'] for s in sample_list]),
        'true_elev_deg': np.array([s.get('true_elev_deg', np.nan) for s in sample_list]),
        'sat':    [s['sat_name'] for s in sample_list],
        'true_pos': np.array([s['true_pos'] for s in sample_list]),
        'true_vel': np.array([s['true_vel'] for s in sample_list]),
    }
    # Fill true_elev_deg for backward compatibility if sample metadata is absent.
    if np.any(~np.isfinite(meta['true_elev_deg'])):
        true_link = ntn_metrics_np(meta['true_pos'], meta['true_vel'],
                                   meta['olat'], meta['olon'], meta['oalt'])
        meta['true_elev_deg'] = true_link['elev']

    preds = {
        'vanilla':  {'pos': np.array([s['sgp4_pos'] for s in sample_list]),
                     'vel': np.array([s['sgp4_vel'] for s in sample_list])},
        'residual': {'pos': np.array([s['baseline_residual_pos'] for s in sample_list]),
                     'vel': np.array([s['baseline_residual_vel'] for s in sample_list])},
        'alongtrack': {'pos': np.array([s['baseline_alongtrack_pos'] for s in sample_list]),
                       'vel': np.array([s['baseline_alongtrack_vel'] for s in sample_list])},
    }
    model.eval()
    mp_all, mv_all = [], []
    dl = DataLoader(DS(sample_list), batch_size=256, shuffle=False, collate_fn=col)
    with torch.no_grad():
        for batch in dl:
            pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
            pred = model(pn, vn, pt, ta, sidx, pr, vr)
            mp_all.append(pred[:, :3].cpu().numpy())
            mv_all.append(pred[:, 3:].cpu().numpy())
    preds['ml_corrected'] = {'pos': np.concatenate(mp_all),
                             'vel': np.concatenate(mv_all)}
    return preds, meta

print('\nRunning all predictors on test set...')
preds, meta = gather_predictions(test_samples)
active_metric_mask = meta['true_elev_deg'] >= cfg.min_eval_elev_deg
if active_metric_mask.sum() == 0:
    print('WARNING: no samples meet visibility mask; falling back to all samples for link-budget summaries.')
    active_metric_mask = np.ones(len(test_samples), dtype=bool)

print(f'  Test samples: {len(test_samples)}')
print(f'  Visible samples used for headline link metrics: {int(active_metric_mask.sum())}')
print(f'  Predictors: {list(preds.keys())}')

# ── (1) PRACH residual Doppler: n255/n256 × IoT-NTN/NR-NTN budgets ──
print('\n' + '='*80)
print('(1) PRACH RESIDUAL DOPPLER vs 3GPP BUDGETS')
print('='*80)

carriers = {
    'n255_DL': NTN_BANDS['n255']['dl_mhz_center']*1e6,
    'n255_UL': NTN_BANDS['n255']['ul_mhz_center']*1e6,
    'n256_DL': NTN_BANDS['n256']['dl_mhz_center']*1e6,
    'n256_UL': NTN_BANDS['n256']['ul_mhz_center']*1e6,
}
predictors = ['vanilla', 'residual', 'alongtrack', 'ml_corrected']
pred_labels = {'vanilla': 'Vanilla SGP4', 'residual': 'Recent-residual',
               'alongtrack': 'Along-track', 'ml_corrected': 'GLASS'}

budget_ref_hz = NTN_3GPP['nr_prach_doppler_budget_hz']
dop_stats = {}
print(f'\n{"Carrier":<10} {"Predictor":<16} {"MAE":>8} {"P95":>8} {"P99":>8} '
      f'{"IoT ok":>8} {"NR ok":>8} {"P95/NR":>9}')
print('-'*86)
for cname, f_hz in carriers.items():
    dop_stats[cname] = {}
    for p in predictors:
        err = compute_doppler_error_hz(preds[p]['pos'], preds[p]['vel'],
                                       meta['true_pos'], meta['true_vel'],
                                       meta['olat'], meta['olon'], meta['oalt'],
                                       f_hz)
        stats = {
            'mae': masked_mean(err, active_metric_mask),
            'p95': masked_percentile(err, 95, active_metric_mask),
            'p99': masked_percentile(err, 99, active_metric_mask),
            'iot_pct': float(np.mean(err[active_metric_mask] < NTN_3GPP['prach_doppler_budget_hz']) * 100),
            'nr_pct': float(np.mean(err[active_metric_mask] < budget_ref_hz) * 100),
            'mae_budget_pct': masked_mean(err, active_metric_mask) / budget_ref_hz * 100,
            'p95_budget_pct': masked_percentile(err, 95, active_metric_mask) / budget_ref_hz * 100,
            'p99_budget_pct': masked_percentile(err, 99, active_metric_mask) / budget_ref_hz * 100,
            'errs': err,
            'mask_used': 'true_elev_deg >= cfg.min_eval_elev_deg',
        }
        dop_stats[cname][p] = stats
        print(f'{cname:<10} {pred_labels[p]:<16} {stats["mae"]:>8.2f} '
              f'{stats["p95"]:>8.2f} {stats["p99"]:>8.2f} '
              f'{stats["iot_pct"]:>7.1f}% {stats["nr_pct"]:>7.1f}% '
              f'{stats["p95_budget_pct"]:>8.1f}%')
    print()

# ── (2) Timing-advance error vs NR CP length ─────────────────────────
print('='*80)
print('(2) TIMING-ADVANCE ERROR vs NR CP LENGTH')
print('='*80)

ta_stats = {}
print(f'\n{"Predictor":<16} {"MAE (us)":>10} {"P95 (us)":>10} {"P99 (us)":>10} '
      f'{"<CP_norm":>10} {"<CP_ext":>10}')
print('-'*72)
for p in predictors:
    ta_err = compute_ta_error_sec(preds[p]['pos'], meta['true_pos'],
                                  meta['olat'], meta['olon'], meta['oalt'])
    ta_err_us = ta_err * 1e6
    stats = {
        'mae_us': masked_mean(ta_err_us, active_metric_mask),
        'p95_us': masked_percentile(ta_err_us, 95, active_metric_mask),
        'p99_us': masked_percentile(ta_err_us, 99, active_metric_mask),
        'ok_norm_pct': float(np.mean(ta_err[active_metric_mask] < NTN_3GPP['nr_cp_normal_15khz_sec']) * 100),
        'ok_ext_pct': float(np.mean(ta_err[active_metric_mask] < NTN_3GPP['nr_cp_extended_15khz_sec']) * 100),
        'errs': ta_err_us,
    }
    ta_stats[p] = stats
    print(f'{pred_labels[p]:<16} {stats["mae_us"]:>10.3f} '
          f'{stats["p95_us"]:>10.3f} {stats["p99_us"]:>10.3f} '
          f'{stats["ok_norm_pct"]:>9.1f}% {stats["ok_ext_pct"]:>9.1f}%')

# ── (3) Beam pointing loss from 2D LOS angular error ─────────────────
print('\n' + '='*80)
print('(3) BEAM POINTING LOSS FROM 2D LOS ERROR (HPBW = {:.2f} deg)'.format(
    NTN_3GPP['ntn_beamwidth_hpbw_deg']))
print('='*80)

pt_stats = {}
print(f'\n{"Predictor":<16} {"LOS MAE":>10} {"Loss MAE":>10} {"Loss P99":>10} '
      f'{"<0.5 dB":>9}')
print(f'{"":<16} {"(deg)":>10} {"(dB)":>10} {"(dB)":>10} {"(%)":>9}')
print('-'*60)
for p in predictors:
    los_err = compute_los_angular_error_deg(preds[p]['pos'], meta['true_pos'],
                                            meta['olat'], meta['olon'], meta['oalt'])
    elev_err = compute_elev_error_deg(preds[p]['pos'], meta['true_pos'],
                                      meta['olat'], meta['olon'], meta['oalt'])
    loss_db = pointing_loss_db(los_err)
    stats = {
        'los_mae_deg': masked_mean(los_err, active_metric_mask),
        'elev_mae_deg': masked_mean(elev_err, active_metric_mask),
        'loss_mae_db': masked_mean(loss_db, active_metric_mask),
        'loss_p99_db': masked_percentile(loss_db, 99, active_metric_mask),
        'ok_pct': float(np.mean(loss_db[active_metric_mask] < 0.5) * 100),
        'errs': loss_db,
        'los_err_deg': los_err,
        'elev_err_deg': elev_err,
    }
    pt_stats[p] = stats
    print(f'{pred_labels[p]:<16} {stats["los_mae_deg"]:>10.4f} '
          f'{stats["loss_mae_db"]:>10.4f} {stats["loss_p99_db"]:>10.4f} '
          f'{stats["ok_pct"]:>8.1f}%')

# ── (4) Horizon curves: n256 DL P95/mean NR-budget consumption ───────
print('\n' + '='*80)
print('(4) NR-NTN DOPPLER-BUDGET CONSUMPTION vs HORIZON (n256 DL)')
print('    Reported as % of the 100 Hz NR-NTN PRACH Doppler budget. Lower is better.')
print('='*80)

horizons = cfg.prop_times
horizon_budget_curves = {
    p: {'mae_budget_pct': [], 'p95_budget_pct': [], 'p99_budget_pct': [], 'availability_nr_pct': []}
    for p in predictors
}
print(f'\n{"Horizon":>7} | ' + ' | '.join(f'{pred_labels[p]:>18}' for p in predictors))
print('-' * (12 + 22 * len(predictors)))
for dt in horizons:
    mask = (meta['dt_min'] == dt) & active_metric_mask
    row = f'{dt:7d}m | '
    for p in predictors:
        if mask.sum() < 5:
            horizon_budget_curves[p]['mae_budget_pct'].append(np.nan)
            horizon_budget_curves[p]['p95_budget_pct'].append(np.nan)
            horizon_budget_curves[p]['p99_budget_pct'].append(np.nan)
            horizon_budget_curves[p]['availability_nr_pct'].append(np.nan)
            row += f'{"-":>18} | '
            continue
        errs = dop_stats['n256_DL'][p]['errs'][mask]
        mae_pct = float(np.mean(errs) / budget_ref_hz * 100)
        p95_pct = float(np.percentile(errs, 95) / budget_ref_hz * 100)
        p99_pct = float(np.percentile(errs, 99) / budget_ref_hz * 100)
        nr_pct = float(np.mean(errs < budget_ref_hz) * 100)
        horizon_budget_curves[p]['mae_budget_pct'].append(mae_pct)
        horizon_budget_curves[p]['p95_budget_pct'].append(p95_pct)
        horizon_budget_curves[p]['p99_budget_pct'].append(p99_pct)
        horizon_budget_curves[p]['availability_nr_pct'].append(nr_pct)
        row += f'{p95_pct:6.1f}% P95 / {mae_pct:5.1f}% MAE | '
    print(row)

# ── (5) Proxy stratification matrix: update-gap × horizon ────────────
print('\n' + '='*80)
print('(5) PROXY STRATIFICATION MATRIX')
print('    P95 n256-DL Doppler-budget consumption (%)')
print('    Binned by: (gap since previous TLE update, propagation horizon)')
print('    Note: this is NOT a stale-TLE replay matrix.')
print('='*80)

gap_bins_lb = [(0,2,'0-2h'),(2,6,'2-6h'),(6,12,'6-12h'),(12,24,'12-24h'),(24,9999,'24h+')]
age_bins_lb = gap_bins_lb  
heatmap = np.full((len(gap_bins_lb), len(horizons)), np.nan)
heatmap_van = np.full((len(gap_bins_lb), len(horizons)), np.nan)
heatmap_counts = np.zeros((len(gap_bins_lb), len(horizons)), dtype=int)
for i, (lo, hi, lbl) in enumerate(gap_bins_lb):
    for j, dt in enumerate(horizons):
        mask = ((meta['inter_update_gap_h'] >= lo) &
                (meta['inter_update_gap_h'] < hi) &
                (meta['dt_min'] == dt) &
                active_metric_mask)
        heatmap_counts[i, j] = int(mask.sum())
        if mask.sum() < 5:
            continue
        ml_err = dop_stats['n256_DL']['ml_corrected']['errs'][mask]
        va_err = dop_stats['n256_DL']['vanilla']['errs'][mask]
        heatmap[i, j]     = float(np.percentile(ml_err, 95) / budget_ref_hz * 100)
        heatmap_van[i, j] = float(np.percentile(va_err, 95) / budget_ref_hz * 100)

print(f'\n  GLASS P95/NR-budget (%):')
print(f'  {"Gap":>10} | ' + ''.join(f'{h:>7}m ' for h in horizons))
print('  ' + '-'*(12 + 9*len(horizons)))
for i, (_,_,lbl) in enumerate(gap_bins_lb):
    row = f'  {lbl:>10} | '
    for j in range(len(horizons)):
        v = heatmap[i,j]
        row += f'{v:6.1f}% ' if not np.isnan(v) else '   -    '
    print(row)

print(f'\n  Vanilla SGP4 P95/NR-budget (%):')
print(f'  {"Gap":>10} | ' + ''.join(f'{h:>7}m ' for h in horizons))
print('  ' + '-'*(12 + 9*len(horizons)))
for i, (_,_,lbl) in enumerate(gap_bins_lb):
    row = f'  {lbl:>10} | '
    for j in range(len(horizons)):
        v = heatmap_van[i,j]
        row += f'{v:6.1f}% ' if not np.isnan(v) else '   -    '
    print(row)

print(f'\n  Samples per stratification cell (N; same mask for vanilla and GLASS):')
print(f'  {"Gap":>10} | ' + ''.join(f'{h:>7}m ' for h in horizons))
print('  ' + '-'*(12 + 9*len(horizons)))
for i, (_,_,lbl) in enumerate(gap_bins_lb):
    row = f'  {lbl:>10} | '
    for j in range(len(horizons)):
        n = heatmap_counts[i, j]
        row += f'{n:7d} ' if n else '   -    '
    print(row)

# Save for figures and audit trail.
def _drop_arrays(stats_dict):
    return {p: {k: v for k, v in s.items()
                if k not in ('errs', 'los_err_deg', 'elev_err_deg')}
            for p, s in stats_dict.items()}

link_budget_results = {
    'dop_stats':  {c: _drop_arrays(ps) for c, ps in dop_stats.items()},
    'ta_stats':   _drop_arrays(ta_stats),
    'pt_stats':   _drop_arrays(pt_stats),
    'horizon_budget_curves_n256_dl': horizon_budget_curves,
    'heatmap':    heatmap.tolist(),
    'heatmap_vanilla': heatmap_van.tolist(),
    'heatmap_counts': heatmap_counts.tolist(),
    'gap_bin_labels': [lbl for _,_,lbl in gap_bins_lb],
    'age_bin_labels': [lbl for _,_,lbl in gap_bins_lb],  # legacy alias
    'horizons': horizons,
    'carriers': carriers,
    'frame_method': globals().get('FRAME_METHOD', 'unknown'),
    'visibility': {
        'min_eval_elev_deg': cfg.min_eval_elev_deg,
        'n_total': int(len(test_samples)),
        'n_used': int(active_metric_mask.sum()),
    },
    'ntn_budgets': {
        'iot_ntn_prach_hz': NTN_3GPP['prach_doppler_budget_hz'],
        'nr_ntn_prach_hz':  NTN_3GPP['nr_prach_doppler_budget_hz'],
        'cp_normal_sec':    NTN_3GPP['nr_cp_normal_15khz_sec'],
        'cp_extended_sec':  NTN_3GPP['nr_cp_extended_15khz_sec'],
        'beamwidth_deg':    NTN_3GPP['ntn_beamwidth_hpbw_deg'],
    },
    'notes': {
        'proxy_stratification': 'Gap since previous TLE update; not a stale-TLE replay experiment.',
        'ta_definition': 'Round-trip timing-advance error: 2 * |range error| / c.',
        'pointing_definition': '2D topocentric LOS angular separation; not elevation-only error.',
        'baseline_definition': 'Causal and horizon-matched recent-residual / along-track baselines.',
        'geometry_definition': 'WGS84 observer plus true geodetic ENU elevation/rate; no geocentric-radial up approximation.',
    },
}
with open('results/link_budget_v9.json', 'w') as f:
    json.dump(link_budget_results, f, indent=2)
# Legacy copy for older figure/analysis scripts.
with open('results/link_budget_v8.json', 'w') as f:
    json.dump(link_budget_results, f, indent=2)
print('\nSaved results/link_budget_v9.json (and legacy copy link_budget_v8.json)')

In [ ]:


print('=' * 88)
print('cached ephemeris / limited-reachback study')
print('=' * 88)

STALE_REPLAY_HOURS = [0, 24, 48, 72, 168]  # fresh/latest, 1d, 2d, 3d, 7d
STALE_REPLAY_MIN_ELEV_DEG = cfg.min_eval_elev_deg
STALE_REPLAY_MAX_ROWS = int(os.environ.get('NTN_STALE_REPLAY_MAX_ROWS', '0'))  # 0 = all visible rows

def _find_cached_tle_at_or_before(sat_name, target_jd, requested_stale_h):
    """Return the freshest TLE with epoch <= target_jd - requested_stale_h.
    requested_stale_h=0 simulates the latest available TLE before target time.
    """
    cutoff = float(target_jd) - float(requested_stale_h) / 24.0
    candidates = [t for t in tle_by_sat.get(sat_name, []) if t['epoch_jd'] <= cutoff]
    return candidates[-1] if candidates else None

def _inter_update_gap_for_tle(sat_name, tle_epoch_jd):
    """Cadence proxy for the selected cached TLE."""
    tles = tle_by_sat.get(sat_name, [])
    prev = [t['epoch_jd'] for t in tles if t['epoch_jd'] < tle_epoch_jd]
    return float((tle_epoch_jd - max(prev)) * 24.0) if prev else 0.0


visible_test_rows = [
    s for s in test_samples
    if float(s.get('true_elev_deg', -999.0)) >= STALE_REPLAY_MIN_ELEV_DEG
]

if STALE_REPLAY_MAX_ROWS > 0 and len(visible_test_rows) > STALE_REPLAY_MAX_ROWS:
    rng = np.random.default_rng(cfg.seed)
    keep_idx = rng.choice(len(visible_test_rows), size=STALE_REPLAY_MAX_ROWS, replace=False)
    visible_test_rows = [visible_test_rows[i] for i in sorted(keep_idx)]

print(
    f'Visible target rows used: {len(visible_test_rows)} '
    f'(true elevation >= {STALE_REPLAY_MIN_ELEV_DEG:.1f} deg)'
)
print(f'Requested stale ages (h): {STALE_REPLAY_HOURS}')


_prop_cache = {}
stale_samples = []
skip_counts = {h: 0 for h in STALE_REPLAY_HOURS}

for s in visible_test_rows:
    sat_name = s['sat_name']
    target_jd = float(s['target_jd'])

    for req_h in STALE_REPLAY_HOURS:
        key = (sat_name, target_jd, int(req_h))

        if key not in _prop_cache:
            cached_tle = _find_cached_tle_at_or_before(sat_name, target_jd, req_h)

            if cached_tle is None:
                _prop_cache[key] = None
            else:
                sp, sv = sgp4_ecef(cached_tle['sat'], target_jd)

                if sp is None:
                    _prop_cache[key] = None
                else:
                    actual_age_h = (target_jd - cached_tle['epoch_jd']) * 24.0
                    update_gap_h = _inter_update_gap_for_tle(sat_name, cached_tle['epoch_jd'])
                    _prop_cache[key] = (cached_tle, sp, sv, actual_age_h, update_gap_h)

        item = _prop_cache[key]

        if item is None:
            skip_counts[req_h] += 1
            continue

        cached_tle, sp, sv, actual_age_h, update_gap_h = item

        true_link = ntn_metrics_np(
            s['true_pos'][None, :],
            s['true_vel'][None, :],
            np.array([s['obs_lat']]),
            np.array([s['obs_lon']]),
            np.array([s['obs_alt']])
        )

        ss = {
            'sgp4_pos': sp.astype(np.float32),
            'sgp4_vel': sv.astype(np.float32),
            'true_pos': s['true_pos'].astype(np.float32),
            'true_vel': s['true_vel'].astype(np.float32),
            'obs_lat': np.float32(s['obs_lat']),
            'obs_lon': np.float32(s['obs_lon']),
            'obs_alt': np.float32(s['obs_alt']),

            
            'prop_time_min': np.float32(actual_age_h * 60.0),

            
            'stale_requested_h': np.float32(req_h),
            'stale_actual_h': np.float32(actual_age_h),

            
            'tle_age_hours': np.float32(update_gap_h),
            'inter_update_gap_hours': np.float32(update_gap_h),
            'tle_propagation_age_hours': np.float32(actual_age_h),

            'true_elev_deg': np.float32(true_link['elev'][0]),
            'is_visible_10deg': bool(true_link['elev'][0] >= cfg.min_eval_elev_deg),
            'dt_min': np.float32(actual_age_h * 60.0),
            'sat_idx': np.int64(SAT_IDX[sat_name]),
            'sat_name': sat_name,
            'tle_epoch_jd': np.float64(cached_tle['epoch_jd']),
            'target_jd': np.float64(target_jd),
        }

        stale_samples.append(ss)

print(f'Constructed stale replay rows: {len(stale_samples)}')
print('Skip counts by requested stale age:', skip_counts)

if not stale_samples:
    raise RuntimeError('No stale replay samples constructed; check TLE coverage and visibility mask.')

# Run gated model on stale-replay samples.
def gather_stale_predictions(sample_list):
    meta_s = {
        'olat': np.array([s['obs_lat'] for s in sample_list]),
        'olon': np.array([s['obs_lon'] for s in sample_list]),
        'oalt': np.array([s['obs_alt'] for s in sample_list]),
        'sat': [s['sat_name'] for s in sample_list],
        'true_pos': np.array([s['true_pos'] for s in sample_list]),
        'true_vel': np.array([s['true_vel'] for s in sample_list]),
        'true_elev_deg': np.array([s['true_elev_deg'] for s in sample_list]),
        'stale_requested_h': np.array([s['stale_requested_h'] for s in sample_list]),
        'stale_actual_h': np.array([s['stale_actual_h'] for s in sample_list]),
        'target_jd': np.array([s['target_jd'] for s in sample_list]),
        'tle_epoch_jd': np.array([s['tle_epoch_jd'] for s in sample_list]),
    }

    pred_s = {
        'vanilla': {
            'pos': np.array([s['sgp4_pos'] for s in sample_list]),
            'vel': np.array([s['sgp4_vel'] for s in sample_list]),
        }
    }

    model.eval()
    mp_all, mv_all = [], []
    dl = DataLoader(DS(sample_list), batch_size=256, shuffle=False, collate_fn=col)

    with torch.no_grad():
        for batch in dl:
            pn, vn, pt, ta, pr, vr, tp, tv, olat, olon, oalt, sidx = [b.to(device) for b in batch]
            pred = model(pn, vn, pt, ta, sidx, pr, vr)
            mp_all.append(pred[:, :3].cpu().numpy())
            mv_all.append(pred[:, 3:].cpu().numpy())

    pred_s['ml_corrected'] = {
        'pos': np.concatenate(mp_all),
        'vel': np.concatenate(mv_all),
    }

    return pred_s, meta_s

stale_preds, stale_meta = gather_stale_predictions(stale_samples)

def _stale_metric_block(predictor_key, mask):
    dop_err = compute_doppler_error_hz(
        stale_preds[predictor_key]['pos'],
        stale_preds[predictor_key]['vel'],
        stale_meta['true_pos'],
        stale_meta['true_vel'],
        stale_meta['olat'],
        stale_meta['olon'],
        stale_meta['oalt'],
        NTN_BANDS['n256']['dl_mhz_center'] * 1e6,
    )

    ta_err_us = compute_ta_error_sec(
        stale_preds[predictor_key]['pos'],
        stale_meta['true_pos'],
        stale_meta['olat'],
        stale_meta['olon'],
        stale_meta['oalt'],
    ) * 1e6

    los_err = compute_los_angular_error_deg(
        stale_preds[predictor_key]['pos'],
        stale_meta['true_pos'],
        stale_meta['olat'],
        stale_meta['olon'],
        stale_meta['oalt'],
    )

    return {
        'dop_mae_hz': float(np.mean(dop_err[mask])),
        'dop_p95_hz': float(np.percentile(dop_err[mask], 95)),
        'dop_p99_hz': float(np.percentile(dop_err[mask], 99)),
        'nr_ok_pct': float(
            np.mean(dop_err[mask] < NTN_3GPP['nr_prach_doppler_budget_hz']) * 100
        ),
        'ta_mae_us': float(np.mean(ta_err_us[mask])),
        'ta_p95_us': float(np.percentile(ta_err_us[mask], 95)),
        'cp_norm_ok_pct': float(
            np.mean((ta_err_us[mask] * 1e-6) < NTN_3GPP['nr_cp_normal_15khz_sec']) * 100
        ),
        'los_mae_deg': float(np.mean(los_err[mask])),
        'n': int(mask.sum()),
    }

stale_replay_results = {
    'stale_hours_requested': STALE_REPLAY_HOURS,
    'visibility_min_deg': STALE_REPLAY_MIN_ELEV_DEG,
    'predictors': ['vanilla', 'ml_corrected'],
    'carrier': 'n256_DL',
    'carrier_hz': NTN_BANDS['n256']['dl_mhz_center'] * 1e6,
    'rows': [],
    'notes': {
        'purpose': 'Cached/stale-TLE replay for limited-reachback / tactical operation.',
        'tle_selection': 'For each target time, choose freshest TLE with epoch <= target - requested_stale_hours.',
        'actual_age': 'Actual propagation age can exceed the requested stale age because TLE cadence is discrete.',
        'scope': 'Visible-link rows only; true elevation >= min_eval_elev_deg.',
        'model_feature': 'Model receives actual TLE propagation age, not only the requested stale-age bin.',
    }
}

print(
    '\nCached-TLE replay summary (true elevation >= {:.1f} deg, n256 DL):'.format(
        STALE_REPLAY_MIN_ELEV_DEG
    )
)
print(
    f'{"Req stale":>10} {"N":>6} {"Actual age med":>14} '
    f'{"Van MAE":>9} {"GLASS MAE":>9} {"Impr":>8} '
    f'{"Van P95":>9} {"GLASS P95":>9} {"NR ok GLASS":>9} '
    f'{"TA P95 GLASS":>10}'
)
print('-' * 112)

for req_h in STALE_REPLAY_HOURS:
    mask = stale_meta['stale_requested_h'] == float(req_h)

    if mask.sum() < 5:
        continue

    actual_age_med = float(np.median(stale_meta['stale_actual_h'][mask]))
    blocks = {}

    for pkey in ['vanilla', 'ml_corrected']:
        blocks[pkey] = _stale_metric_block(pkey, mask)

    impr = (
        (blocks['vanilla']['dop_mae_hz'] - blocks['ml_corrected']['dop_mae_hz'])
        / blocks['vanilla']['dop_mae_hz']
        * 100.0
    )

    row = {
        'requested_stale_h': float(req_h),
        'actual_age_h_median': actual_age_med,
        'actual_age_h_p95': float(np.percentile(stale_meta['stale_actual_h'][mask], 95)),
        'vanilla': blocks['vanilla'],
        'ml_corrected': blocks['ml_corrected'],
        'dop_mae_improvement_pct': float(impr),
    }

    stale_replay_results['rows'].append(row)

    print(
        f'{req_h:10.0f} {blocks["vanilla"]["n"]:6d} {actual_age_med:14.1f} '
        f'{blocks["vanilla"]["dop_mae_hz"]:9.2f} {blocks["ml_corrected"]["dop_mae_hz"]:9.2f} '
        f'{impr:7.1f}% {blocks["vanilla"]["dop_p95_hz"]:9.2f} '
        f'{blocks["ml_corrected"]["dop_p95_hz"]:9.2f} '
        f'{blocks["ml_corrected"]["nr_ok_pct"]:8.1f}% '
        f'{blocks["ml_corrected"]["ta_p95_us"]:10.2f}'
    )

# Save JSON and CSV.
with open('results/stale_tle_replay_v9.json', 'w') as f:
    json.dump(stale_replay_results, f, indent=2)

try:
    import pandas as pd

    flat_rows = []

    for r in stale_replay_results['rows']:
        out = {
            'requested_stale_h': r['requested_stale_h'],
            'actual_age_h_median': r['actual_age_h_median'],
            'actual_age_h_p95': r['actual_age_h_p95'],
            'dop_mae_improvement_pct': r['dop_mae_improvement_pct'],
        }

        for pkey in ['vanilla', 'ml_corrected']:
            for k, v in r[pkey].items():
                out[f'{pkey}_{k}'] = v

        flat_rows.append(out)

    pd.DataFrame(flat_rows).to_csv('results/stale_tle_replay_v9.csv', index=False)
    print('Saved results/stale_tle_replay_v9.csv')

except Exception as ex:
    print(f'CSV save skipped: {ex}')


C_VAN = globals().get('C_VAN', '#D35400')
C_ML = globals().get('C_ML', '#2471A3')
C_BUDG = globals().get('C_BUDG', '#C0392B')
TITLE_SIZE = globals().get('TITLE_SIZE', 12)
LABEL_SIZE = globals().get('LABEL_SIZE', 10)
TICK_SIZE = globals().get('TICK_SIZE', 9)
ANNOT_SIZE = globals().get('ANNOT_SIZE', 8)
DPI = globals().get('DPI', 300)

fig, ax = plt.subplots(figsize=(8.5, 5.0), constrained_layout=True)

x = np.array([r['requested_stale_h'] for r in stale_replay_results['rows']], dtype=float)
x_plot = np.where(x == 0, 0.5, x)  

for pkey, label, color, marker in [
    ('vanilla', 'Cached stale-SGP4', C_VAN, 's'),
    ('ml_corrected', 'GLASS', C_ML, 'o'),
]:
    p95_budget = np.array([
        r[pkey]['dop_p95_hz'] / NTN_3GPP['nr_prach_doppler_budget_hz'] * 100.0
        for r in stale_replay_results['rows']
    ])

    ax.plot(
        x_plot,
        p95_budget,
        marker=marker,
        lw=2.5,
        color=color,
        label=label,
    )

ax.axhline(
    100,
    color=C_BUDG,
    ls=':',
    lw=1.5,
    label='NR-NTN PRACH budget',
)

ax.set_xscale('log')
ax.set_xticks(x_plot)
ax.set_xticklabels(['fresh' if h == 0 else f'{int(h)}h' for h in x])
ax.set_xlabel('Requested cached-TLE staleness', fontsize=LABEL_SIZE)
ax.set_ylabel('P95 n256-DL Doppler error / NR budget (%)', fontsize=LABEL_SIZE)
ax.grid(alpha=0.25, which='both')
ax.legend(fontsize=TICK_SIZE)

for r, xp in zip(stale_replay_results['rows'], x_plot):
    y_val = (
        r['ml_corrected']['dop_p95_hz']
        / NTN_3GPP['nr_prach_doppler_budget_hz']
        * 100.0
    )

    ax.annotate(
        f'N={r["vanilla"]["n"]}',
        xy=(xp, y_val),
        xytext=(0, 8),
        textcoords='offset points',
        ha='center',
        fontsize=ANNOT_SIZE,
    )

fig.savefig('results/09_stale_tle_replay.png', dpi=DPI)
fig.savefig('results/09_stale_tle_replay.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

print('Saved results/stale_tle_replay_v9.json')
print('Saved results/09_stale_tle_replay.png and results/09_stale_tle_replay.pdf')

In [ ]:


import csv
from collections import defaultdict

print('=' * 80)

print('=' * 80)

_rng = np.random.default_rng(cfg.bootstrap_seed)


block_keys = np.array([
    f'{sat}|{tjd:.8f}|{int(dt)}'
    for sat, tjd, dt in zip(meta['sat'], meta['target_jd'], meta['dt_min'])
])

def _block_bootstrap_ci(values, mask, stat_fn, n_boot=None, alpha=0.05):
    values = np.asarray(values, dtype=float)
    mask = np.asarray(mask, dtype=bool)
    n_boot = cfg.bootstrap_n if n_boot is None else int(n_boot)
    valid_idx = np.where(mask & np.isfinite(values))[0]
    if valid_idx.size == 0:
        return {'point': np.nan, 'lo': np.nan, 'hi': np.nan, 'n_rows': 0, 'n_blocks': 0}
    valid_blocks = np.unique(block_keys[valid_idx])
    block_to_idx = {b: valid_idx[block_keys[valid_idx] == b] for b in valid_blocks}
    point = float(stat_fn(values[valid_idx]))
    if len(valid_blocks) < 2 or n_boot <= 0:
        return {'point': point, 'lo': np.nan, 'hi': np.nan,
                'n_rows': int(valid_idx.size), 'n_blocks': int(len(valid_blocks))}
    boot = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        chosen = _rng.choice(valid_blocks, size=len(valid_blocks), replace=True)
        idx = np.concatenate([block_to_idx[c] for c in chosen])
        boot[b] = float(stat_fn(values[idx]))
    return {
        'point': point,
        'lo': float(np.percentile(boot, 100 * alpha / 2)),
        'hi': float(np.percentile(boot, 100 * (1 - alpha / 2))),
        'n_rows': int(valid_idx.size),
        'n_blocks': int(len(valid_blocks)),
    }

def _fmt_ci(d, units=''):
    if not np.isfinite(d['lo']):
        return f"{d['point']:.3f}{units}"
    return f"{d['point']:.3f}{units} [{d['lo']:.3f}, {d['hi']:.3f}]"

headline_rows = []
metric_specs = []
for p in predictors:
    metric_specs.append(('Doppler MAE n256 DL (Hz)', p, dop_stats['n256_DL'][p]['errs'], np.mean, 'Hz'))
    metric_specs.append(('Doppler P95 n256 DL (Hz)', p, dop_stats['n256_DL'][p]['errs'], lambda x: np.percentile(x, 95), 'Hz'))
    metric_specs.append(('Timing-advance MAE (us)', p, ta_stats[p]['errs'], np.mean, 'us'))
    metric_specs.append(('Timing-advance P95 (us)', p, ta_stats[p]['errs'], lambda x: np.percentile(x, 95), 'us'))
    metric_specs.append(('Beam-loss MAE (dB)', p, pt_stats[p]['errs'], np.mean, 'dB'))
    metric_specs.append(('LOS pointing P95 (deg)', p, pt_stats[p]['los_err_deg'], lambda x: np.percentile(x, 95), 'deg'))

for metric, p, values, stat_fn, units in metric_specs:
    ci = _block_bootstrap_ci(values, active_metric_mask, stat_fn)
    headline_rows.append({
        'metric': metric,
        'predictor': p,
        'predictor_label': pred_labels[p],
        'point': ci['point'],
        'ci95_lo': ci['lo'],
        'ci95_hi': ci['hi'],
        'units': units,
        'n_rows': ci['n_rows'],
        'n_blocks': ci['n_blocks'],
        'visibility_min_deg': cfg.min_eval_elev_deg,
    })


strongest_rows = []
for metric in sorted(set(r['metric'] for r in headline_rows)):
    rows = [r for r in headline_rows if r['metric'] == metric]
    ml = next(r for r in rows if r['predictor'] == 'ml_corrected')
    base_candidates = [r for r in rows if r['predictor'] in ('vanilla', 'residual', 'alongtrack')]
    best_base = min(base_candidates, key=lambda r: r['point'])
    improvement_pct = (best_base['point'] - ml['point']) / best_base['point'] * 100 if best_base['point'] > 0 else np.nan
    strongest_rows.append({
        'metric': metric,
        'strongest_baseline': best_base['predictor_label'],
        'baseline_point': best_base['point'],
        'ml_point': ml['point'],
        'ml_vs_strongest_improvement_pct': improvement_pct,
        'units': ml['units'],
        'n_rows': ml['n_rows'],
        'n_blocks': ml['n_blocks'],
    })

print('\nHeadline metrics with 95% block-bootstrap CIs:')
for r in headline_rows:
    if r['predictor'] not in ('vanilla', 'ml_corrected'):
        continue
    print(f"  {r['metric']:<32} {r['predictor_label']:<16} "
          f"{r['point']:.3f} [{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}] {r['units']} "
          f"(N={r['n_rows']}, blocks={r['n_blocks']})")

print('\nGLASS vs strongest causal baseline:')
for r in strongest_rows:
    print(f"  {r['metric']:<32} base={r['strongest_baseline']:<16} "
          f"GLASS improvement={r['ml_vs_strongest_improvement_pct']:+.1f}%")

os.makedirs('results', exist_ok=True)
with open('results/ci_v8.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(headline_rows[0].keys()))
    writer.writeheader(); writer.writerows(headline_rows)
with open('results/strongest_baseline_headline_v8.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(strongest_rows[0].keys()))
    writer.writeheader(); writer.writerows(strongest_rows)
with open('results/ci_v8.json', 'w') as f:
    json.dump({
        'bootstrap': {
            'method': 'block bootstrap',
            'block_key': 'satellite|target_jd|propagation_horizon',
            'n_boot': cfg.bootstrap_n,
            'seed': cfg.bootstrap_seed,
        },
        'visibility_min_deg': cfg.min_eval_elev_deg,
        'headline_rows': headline_rows,
        'strongest_baseline_rows': strongest_rows,
    }, f, indent=2)

print('\nSaved results/ci_v8.csv')
print('Saved results/strongest_baseline_headline_v8.csv')
print('Saved results/ci_v8.json')

In [ ]:


torch.save({
    'model_state_dict': model.state_dict(),
    'hidden_dim': cfg.hidden, 'n_layers': cfg.layers,
    'input_dim_nonembed': 8,
    'sat_embed_dim': cfg.sat_embed_dim,
    'n_sats': N_SATS,
    'sat_idx_map': SAT_IDX,
    'architecture': model.__class__.__name__,
    'version': 'stale_replay',
    'frame_method': globals().get('FRAME_METHOD', 'unknown'),
    'iers_source': globals().get('IERS_SOURCE', 'unknown'),
    'iers_sha256': globals().get('IERS_SHA256', None),
    'paper_mode': cfg.paper_mode,
    'best_ablation': best_name,
    'best_loss_cfg': best_loss_cfg,
    'selection_criteria': {
        'primary_metrics': primary_ntn_metrics,
        'baseline_reference': 'strongest per-metric among vanilla, recent-residual, along-track',
        'tolerance_vs_ops': '2% on NTN metrics',
        'position_sanity': 'model position MAE <= 1.05 * vanilla position MAE',
    },
    'gate_calibration_metrics': globals().get('gate_calibration_metrics', None),
    'gate_config': {
        'gate_hidden': cfg.gate_hidden,
        'gate_epochs': cfg.gate_epochs,
        'gate_lr': cfg.gate_lr,
        'gate_min_pos_alpha': cfg.gate_min_pos_alpha,
        'gate_min_vel_alpha': cfg.gate_min_vel_alpha,
        'max_pos_correction_km': cfg.max_pos_correction_km,
        'max_vel_correction_kms': cfg.max_vel_correction_kms,
        'pos_safety_margin_km': cfg.pos_safety_margin_km,
        'delay_safety_margin_us': cfg.delay_safety_margin_us,
        'gate_safety_weight': cfg.gate_safety_weight,
        'gate_delay_safety_weight': cfg.gate_delay_safety_weight,
        'gate_l1_weight': cfg.gate_l1_weight,
    },
    'loss_scales': loss_scales,
    'normalization': {'r_norm':cfg.r_norm,'v_norm':cfg.v_norm,
                      't_norm':cfg.t_norm,'age_norm':cfg.age_norm},
    'training': {'epochs':cfg.epochs,'lr':cfg.lr,'lr_min':cfg.lr_min,
                 'seed': cfg.seed,
                 'satellites': list(precise_data.keys()),
                 'split': 'temporal',
                 'train_end': TRAIN_END,
                 'prop_times': cfg.prop_times,
                 'n_train': len(train_set), 'n_val': len(val_set),
                 'n_test': len(test_samples)},
    'evaluation': {
        'min_eval_elev_deg': cfg.min_eval_elev_deg,
        'visibility_cuts_deg': list(cfg.visibility_cuts_deg),
                 'geometry': 'WGS84 observer position + geodetic ENU elevation/rate',
        'visible_eval': visibility_eval,
        'strongest_baseline_metrics': strongest_baseline_metrics,
        'operational_baseline_metrics': operational_baseline_metrics,
    },
    'test_results': rm,
}, 'results/ntn_model_final.pt')

results_payload = {
    'vanilla': rv,
    'model': rm,
    'operational_baseline_metrics': operational_baseline_metrics,
    'strongest_baseline_metrics': strongest_baseline_metrics,
    'ablation_results': ablation_results,
    'best_ablation': best_name,
    'best_loss_cfg': best_loss_cfg,
    'gate_calibration_metrics': globals().get('gate_calibration_metrics', None),
    'gate_config': {
        'gate_hidden': cfg.gate_hidden,
        'gate_epochs': cfg.gate_epochs,
        'gate_lr': cfg.gate_lr,
        'gate_min_pos_alpha': cfg.gate_min_pos_alpha,
        'gate_min_vel_alpha': cfg.gate_min_vel_alpha,
        'max_pos_correction_km': cfg.max_pos_correction_km,
        'max_vel_correction_kms': cfg.max_vel_correction_kms,
        'pos_safety_margin_km': cfg.pos_safety_margin_km,
        'delay_safety_margin_us': cfg.delay_safety_margin_us,
        'gate_safety_weight': cfg.gate_safety_weight,
        'gate_delay_safety_weight': cfg.gate_delay_safety_weight,
        'gate_l1_weight': cfg.gate_l1_weight,
    },
    'loss_scales': loss_scales,
    'gate_calibration_metrics': globals().get('gate_calibration_metrics', None),
    'visibility_eval': visibility_eval,
    'stale_tle_replay': globals().get('stale_replay_results', None),
    'config': {'hidden': cfg.hidden, 'layers': cfg.layers,
               'epochs': cfg.epochs, 'seed': cfg.seed,
               'sat_embed_dim': cfg.sat_embed_dim,
               'prop_times': cfg.prop_times,
               't_norm': cfg.t_norm,
               'satellites': list(precise_data.keys()),
               'sat_idx_map': SAT_IDX,
               'frame_method': globals().get('FRAME_METHOD', 'unknown'),
    'iers_source': globals().get('IERS_SOURCE', 'unknown'),
    'iers_sha256': globals().get('IERS_SHA256', None),
    'paper_mode': cfg.paper_mode,
               'min_eval_elev_deg': cfg.min_eval_elev_deg,
               'split': 'temporal', 'train_end': TRAIN_END,
               'n_train': len(train_set), 'n_val': len(val_set),
               'n_test': len(test_samples)}
}

with open('results/results_v9.json', 'w') as f:
    json.dump(results_payload, f, indent=2)


with open('results/results_v3.json', 'w') as f:
    json.dump(results_payload, f, indent=2)

with open('results/results_by_dt_v9.json', 'w') as f:
    json.dump({'vanilla': {str(k): v for k, v in dtv.items()},
               'model':   {str(k): v for k, v in dtm.items()},
               'min_eval_elev_deg': cfg.min_eval_elev_deg}, f, indent=2)

print('Saved:')
print('  results/ntn_model_final.pt        <- deploy/reload checkpoint')
print('  results/results_v9.json           <- all ablation + baseline results')
print('  results/results_by_dt_v9.json     <- visibility-filtered per-horizon breakdown')
print('  results/link_budget_v9.json       <- link-budget results from Cell 4.5')
print(f'\nModel architecture: {model.__class__.__name__}')
print(f'Trained satellites: {list(precise_data.keys())}')
print(f'Best ablation:      {best_name} with {best_loss_cfg}')
print(f'Frame method:       {globals().get("FRAME_METHOD", "unknown")}')
print(f'Temporal split:     train < {TRAIN_END}, test >= {TRAIN_END}')

In [ ]:


import platform, sys

manifest = {
    'project': 'GLASS for 3GPP NR-NTN Link Planning',
    'notebook_version': 'v9',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'device': str(device),
    'frame_method': globals().get('FRAME_METHOD', 'unknown'),
    'iers_source': globals().get('IERS_SOURCE', 'unknown'),
    'iers_sha256': globals().get('IERS_SHA256', None),
    'paper_mode': cfg.paper_mode,
    'allow_gmst_fallback': cfg.allow_gmst_fallback,
    'date_range': {'start': START_DATE, 'end': END_DATE,
                   'train_end': TRAIN_END, 'test_start': TEST_START},
    'satellites_requested': SATELLITES,
    'satellites_loaded': list(precise_data.keys()),
    'sample_counts': {'train': len(train_set), 'val': len(val_set), 'test': len(test_samples)},
    'propagation_horizons_min': cfg.prop_times,
    'observer_grid': {
        'lat_range_deg': [-60, 60],
        'lat_count': 15,
        'lon_count': 24,
        'obs_per_sample': cfg.obs_per_sample,
    },
    'visibility': {
        'headline_min_elevation_deg': cfg.min_eval_elev_deg,
        'cuts_deg': list(cfg.visibility_cuts_deg),
    },
    'model': {
        'architecture': model.__class__.__name__,
        'hidden': cfg.hidden,
        'layers': cfg.layers,
        'sat_embed_dim': cfg.sat_embed_dim,
        'n_params': int(model.n_params()),
        'best_ablation': best_name,
        'best_loss_cfg': best_loss_cfg,
        'gate_config': {
            'gate_hidden': cfg.gate_hidden,
            'gate_epochs': cfg.gate_epochs,
            'max_pos_correction_km': cfg.max_pos_correction_km,
            'max_vel_correction_kms': cfg.max_vel_correction_kms,
            'pos_safety_margin_km': cfg.pos_safety_margin_km,
            'delay_safety_margin_us': cfg.delay_safety_margin_us,
        },
        'gate_calibration_metrics': globals().get('gate_calibration_metrics', None),
    },
    'stale_tle_replay': globals().get('stale_replay_results', None),
    'baselines': {
        'vanilla': 'SGP4 propagated from selected TLE',
        'recent_residual': 'causal horizon-matched mean POD-SGP4 residual',
        'alongtrack': 'causal horizon-matched along-track residual projection',
        'selection_reference': 'strongest per-metric causal operational baseline',
    },
    'credential_note': 'Earthdata, VirES, and Space-Track credentials are read from environment variables or prompts; never saved.',
}
with open('results/repro_manifest_v9.json', 'w') as f:
    json.dump(manifest, f, indent=2)

readme = f"""# Reproducing  NTN GLASS experiment

1. Set credentials with environment variables:
   - `EARTHDATA_TOKEN`
   - `VIRES_TOKEN`
   - `SPACETRACK_USER`
   - `SPACETRACK_PASS`

2. Optional: set `NTN_CACHE_DIR` to reuse large Sentinel-3A `.npy` files.

3. Run cells in order.`.
   If the notebook prints `gmst_fallback`, the run is suitable for debugging but
   not for final claims.

4. Headline link-budget results use true elevation >= {cfg.min_eval_elev_deg:.1f}°.
   The all-geometry metrics are retained for diagnostics only.

5. Main artifacts:
   - `results/ntn_model_final.pt`
   - `results/results_v9.json`
   - `results/results_by_dt_v9.json`
   - `results/link_budget_v9.json`
   - `results/repro_manifest_v9.json`
   - `results/stale_tle_replay_v9.json`
   - `results/stale_tle_replay_v9.csv`
   - `results/09_stale_tle_replay.png` and `results/09_stale_tle_replay.pdf`
"""
with open('results/README_reproduce_v9.md', 'w') as f:
    f.write(readme)

print('Saved reproducibility artifacts:')
print('  results/repro_manifest_v9.json')
print('  results/README_reproduce_v9.md')

In [ ]:

@torch.no_grad()
def compute_all_errors(mdl, loader):
    """Compute per-sample errors for all metrics. Returns dict of arrays."""
    if mdl: mdl.eval(); mdl = mdl.to(device)
    errs = {k: [] for k in ['dop','elev','delay','rate','pos']}
    for batch in loader:
        pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
        pred = mdl(pn,vn,pt,ta,sidx,pr,vr) if mdl else torch.cat([pr,vr],dim=-1)
        true = torch.cat([tp,tv],dim=-1)
        p = ntn_params(pred[:,:3],pred[:,3:],olat,olon,oalt)
        t = ntn_params(true[:,:3],true[:,3:],olat,olon,oalt)
        errs['dop'].append(torch.abs(p['dop']-t['dop']).cpu().numpy())
        errs['elev'].append(torch.abs(p['elev']-t['elev']).cpu().numpy())
        errs['delay'].append(torch.abs(p['delay']-t['delay']).cpu().numpy())
        errs['rate'].append(torch.abs(p['rate']-t['rate']).cpu().numpy())
        errs['pos'].append(torch.norm(pred[:,:3]-true[:,:3],dim=-1).cpu().numpy())
    return {k: np.concatenate(v) for k,v in errs.items()}

van_errs = compute_all_errors(None, test_loader)
mod_errs = compute_all_errors(model, test_loader)

pcts = [50, 90, 95, 99]
metric_labels = {'dop': 'Doppler (Hz)', 'elev': 'Elevation (deg)',
                 'delay': 'Delay (us)', 'rate': 'Elev Rate (d/s)', 'pos': 'Position (km)'}

print('=' * 90)
print('TASK 1: TAIL PERCENTILE ANALYSIS')
print('=' * 90)
print('\n(a) Overall Percentiles — Vanilla vs Model\n')
print(f'{"Metric":>20} {"Source":>8} {"MAE":>9} {"P50":>9} {"P90":>9} {"P95":>9} {"P99":>9} {"Max":>9} {"P99/MAE":>8}')
print('-' * 90)

heavy_tail_flags = []
for key, label in metric_labels.items():
    for name, data in [('Vanilla', van_errs[key]), ('Model', mod_errs[key])]:
        mae = np.mean(data)
        p = [np.percentile(data, q) for q in pcts]
        mx = np.max(data)
        ratio = p[3] / mae if mae > 0 else 0
        flag = ' !' if ratio > 5 else ''
        print(f'{label:>20} {name:>8} {mae:>9.4f} {p[0]:>9.4f} {p[1]:>9.4f} {p[2]:>9.4f} {p[3]:>9.4f} {mx:>9.4f} {ratio:>7.1f}x{flag}')
        if ratio > 5:
            heavy_tail_flags.append(f'{label} ({name})')
    print()

if heavy_tail_flags:
    print(f'Heavy tails detected (P99 > 5x MAE): {", ".join(heavy_tail_flags)}')
    print('  Consider quantile regression heads or deep ensembles for calibrated uncertainty.')
else:
    print('No heavy tails detected (all P99 < 5x MAE)')


print('\n\n(b) Position Error Percentiles by Propagation Time\n')
print(f'{"dt (min)":>8} {"Source":>8} {"MAE":>8} {"P50":>8} {"P90":>8} {"P95":>8} {"P99":>8} {"Max":>8}')
print('-' * 72)

for dt in cfg.prop_times:
    dt_idx = [i for i, s in enumerate(test_samples) if s['dt_min'] == dt]
    if not dt_idx: continue
    ds = DS([test_samples[i] for i in dt_idx])
    dl = DataLoader(ds, batch_size=256, shuffle=False, collate_fn=col)
    v = compute_all_errors(None, dl)
    m = compute_all_errors(model, dl)
    for name, data in [('Vanilla', v['pos']), ('Model', m['pos'])]:
        mae = np.mean(data)
        p = [np.percentile(data, q) for q in pcts]
        print(f'{dt:>8} {name:>8} {mae:>8.4f} {p[0]:>8.4f} {p[1]:>8.4f} {p[2]:>8.4f} {p[3]:>8.4f} {np.max(data):>8.4f}')
    print()

In [ ]:

print('=' * 90)
print('TASK 2: PROXY STRATIFICATION BY GAP SINCE PREVIOUS TLE UPDATE')
print('=' * 90)

gap_bins = [(0, 2, '0-2h'), (2, 6, '2-6h'), (6, 12, '6-12h'), (12, 24, '12-24h'), (24, 9999, '24h+')]

print(f'\n{"Bin":>8} {"N":>7} {"Metric":>12} {"Vanilla":>10} {"Model":>10} {"Impr":>8}')
print('-' * 60)

any_drop = False
for lo, hi, label in gap_bins:
    bin_samples = [s for s in test_samples if lo <= s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)) < hi]
    n = len(bin_samples)
    if n < 10:
        print(f'{label:>8} {n:>7}   (too few samples, skipping)')
        continue
    ds = DS(bin_samples)
    dl = DataLoader(ds, batch_size=256, shuffle=False, collate_fn=col)
    v = compute_all_errors(None, dl)
    m = compute_all_errors(model, dl)
    first = True
    for key, mname in [('pos', 'Position(km)'), ('dop', 'Doppler(Hz)'), ('delay', 'Delay(us)')]:
        vm, mm = np.mean(v[key]), np.mean(m[key])
        impr = (vm - mm) / vm * 100 if vm > 0 else 0
        tag = label if first else ''
        ntag = str(n) if first else ''
        flag = ' !' if impr < 5 else ''
        print(f'{tag:>8} {ntag:>7} {mname:>12} {vm:>10.4f} {mm:>10.4f} {impr:>+7.1f}%{flag}')
        if impr < 5:
            any_drop = True
        first = False
    print()

print('Interpretation:')
if any_drop:
    print('Improvement < 5% for some inter-update-gap bins — check sample balance or look for'
          '\n  systematic structure in those bins.')
else:
    print('Model improves consistently across all inter-update-gap bins in this temporal holdout.')

ages = [s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)) for s in test_samples]
print(f'\nProxy-gap distribution in test set:')
print(f'  Mean: {np.mean(ages):.1f}h, Median: {np.median(ages):.1f}h, '
      f'Min: {np.min(ages):.1f}h, Max: {np.max(ages):.1f}h')
for lo, hi, label in gap_bins:
    n = sum(1 for a in ages if lo <= a < hi)
    print(f'  {label}: {n} samples ({n/len(ages)*100:.1f}%)')
print('\nNote: this is a proxy stratification by time since previous TLE update, not a stale-TLE replay study.')

In [ ]:

print('=' * 90)
print('TASK 3: PHYSICAL CONSISTENCY CHECKS')
print('=' * 90)

@torch.no_grad()
def collect_states_and_meta(mdl, samples):
    """Run a predictor and return pos/vel plus truth and satellite metadata."""
    if mdl: mdl.eval(); mdl = mdl.to(device)
    dl = DataLoader(DS(samples), batch_size=256, shuffle=False, collate_fn=col)
    all_pos, all_vel, all_true_pos, all_true_vel = [], [], [], []
    for batch in dl:
        pn,vn,pt,ta,pr,vr,tp,tv,olat,olon,oalt,sidx = [b.to(device) for b in batch]
        pred = mdl(pn,vn,pt,ta,sidx,pr,vr) if mdl else torch.cat([pr,vr],dim=-1)
        all_pos.append(pred[:,:3].cpu().numpy())
        all_vel.append(pred[:,3:].cpu().numpy())
        all_true_pos.append(tp.cpu().numpy())
        all_true_vel.append(tv.cpu().numpy())
    pos = np.concatenate(all_pos); vel = np.concatenate(all_vel)
    true_pos = np.concatenate(all_true_pos); true_vel = np.concatenate(all_true_vel)
    sat_names = np.array([s['sat_name'] for s in samples])
    return pos, vel, true_pos, true_vel, sat_names

van_pos, van_vel, true_pos, true_vel, sat_names = collect_states_and_meta(None, test_samples)
mod_pos, mod_vel, _, _, _ = collect_states_and_meta(model, test_samples)


print('\n(a) Per-Satellite Altitude Error vs Precise-Orbit Truth\n')
print(f'{"Satellite":>15} {"N":>7} {"Van MAE":>10} {"Model MAE":>10} {"Model P95":>10} {"Impr":>8}')
print('-' * 72)

any_bad = False
for sat_name in sorted(SAT_IDX.keys()):
    mask = sat_names == sat_name
    if mask.sum() == 0: continue
    true_alt = np.linalg.norm(true_pos[mask], axis=1) - cfg.re
    v_alt = np.linalg.norm(van_pos[mask], axis=1) - cfg.re
    m_alt = np.linalg.norm(mod_pos[mask], axis=1) - cfg.re
    v_err = np.abs(v_alt - true_alt)
    m_err = np.abs(m_alt - true_alt)
    impr = (np.mean(v_err) - np.mean(m_err)) / np.mean(v_err) * 100 if np.mean(v_err) > 0 else np.nan
    print(f'{sat_name:>15} {mask.sum():>7} {np.mean(v_err):>10.3f} {np.mean(m_err):>10.3f} '
          f'{np.percentile(m_err,95):>10.3f} {impr:>+7.1f}%')
    if np.percentile(m_err, 99) > 10.0:
        any_bad = True
        print(f'  ! Model altitude P99 error exceeds 10 km for {sat_name}')

underground = np.sum(np.linalg.norm(mod_pos, axis=1) - cfg.re < 0)
if underground > 0:
    print(f'\nFAIL: {underground} corrected positions are underground (impossible)!')
elif not any_bad:
    print('\nPASS: Model altitude remains close to precise-orbit truth; no nominal-altitude assumption used')


print('\n(b) Per-Satellite Velocity-Magnitude Error vs Precise-Orbit Truth\n')
print(f'{"Satellite":>15} {"N":>7} {"Van MAE":>12} {"Model MAE":>12} {"Model P95":>12} {"Impr":>8}')
print('-' * 80)
for sat_name in sorted(SAT_IDX.keys()):
    mask = sat_names == sat_name
    if mask.sum() == 0: continue
    true_v = np.linalg.norm(true_vel[mask], axis=1)
    vv = np.linalg.norm(van_vel[mask], axis=1)
    mv = np.linalg.norm(mod_vel[mask], axis=1)
    v_err = np.abs(vv - true_v)
    m_err = np.abs(mv - true_v)
    impr = (np.mean(v_err) - np.mean(m_err)) / np.mean(v_err) * 100 if np.mean(v_err) > 0 else np.nan
    print(f'{sat_name:>15} {mask.sum():>7} {np.mean(v_err):>12.6f} {np.mean(m_err):>12.6f} '
          f'{np.percentile(m_err,95):>12.6f} {impr:>+7.1f}%')


print('\n\n(c) Correction Magnitude vs SGP4 Error\n')
van_errors = np.linalg.norm(van_pos - true_pos, axis=1)
model_errors = np.linalg.norm(mod_pos - true_pos, axis=1)
corr_mag = np.linalg.norm(mod_pos - van_pos, axis=1)
print(f'  Mean correction magnitude: {np.mean(corr_mag):.4f} km')
print(f'  Mean SGP4 error:           {np.mean(van_errors):.4f} km')
print(f'  Mean model error:          {np.mean(model_errors):.4f} km')
ratio = np.mean(corr_mag) / np.mean(van_errors) if np.mean(van_errors) > 0 else 0
print(f'  Ratio (correction / SGP4 error): {ratio:.2f}x')
if ratio > 3:
    print('  ! Corrections substantially larger than errors — possible overcorrection')
elif ratio < 0.3:
    print('  ! Corrections very small — model may be underfitting')
else:
    print('  PASS: correction magnitudes proportionate to errors')


print('\n\n(d) Angular Momentum and Specific Orbital Energy vs Truth\n')
MU_EARTH_KM3_S2 = 398600.4418

def _h(pos, vel):
    return np.linalg.norm(np.cross(pos, vel), axis=1)

def _energy(pos, vel):
    return 0.5*np.sum(vel*vel, axis=1) - MU_EARTH_KM3_S2 / np.linalg.norm(pos, axis=1)

true_L = _h(true_pos, true_vel); van_L = _h(van_pos, van_vel); mod_L = _h(mod_pos, mod_vel)
true_E = _energy(true_pos, true_vel); van_E = _energy(van_pos, van_vel); mod_E = _energy(mod_pos, mod_vel)
L_v_err = np.abs(van_L - true_L) / np.abs(true_L)
L_m_err = np.abs(mod_L - true_L) / np.abs(true_L)
E_v_err = np.abs(van_E - true_E) / np.maximum(np.abs(true_E), 1e-12)
E_m_err = np.abs(mod_E - true_E) / np.maximum(np.abs(true_E), 1e-12)
print(f'  Angular momentum rel. error: vanilla mean={np.mean(L_v_err)*100:.4f}%, model mean={np.mean(L_m_err)*100:.4f}%')
print(f'  Angular momentum rel. error: vanilla P95 ={np.percentile(L_v_err,95)*100:.4f}%, model P95 ={np.percentile(L_m_err,95)*100:.4f}%')
print(f'  Specific energy rel. error:  vanilla mean={np.mean(E_v_err)*100:.4f}%, model mean={np.mean(E_m_err)*100:.4f}%')
print(f'  Specific energy rel. error:  vanilla P95 ={np.percentile(E_v_err,95)*100:.4f}%, model P95 ={np.percentile(E_m_err,95)*100:.4f}%')
if np.mean(L_m_err) < max(np.mean(L_v_err)*1.10, 1e-6) and np.mean(E_m_err) < max(np.mean(E_v_err)*1.10, 1e-6):
    print('  PASS: model does not materially degrade angular momentum or energy relative to truth')
else:
    print('  CHECK: model orbital invariants should be reviewed for this run')

In [ ]:


C_VAN = '#D35400'   # vanilla SGP4
C_RES = '#F4A261'   # recent-residual baseline
C_ATR = '#E9C46A'   # along-track baseline
C_ML  = '#2471A3'   # GLASS
C_BUDG= '#C0392B'   # budget line

PRED_COLORS = {'vanilla': C_VAN, 'residual': C_RES,
               'alongtrack': C_ATR, 'ml_corrected': C_ML}
PRED_LABELS = {'vanilla': 'Vanilla SGP4', 'residual': 'Recent-residual',
               'alongtrack': 'Along-track', 'ml_corrected': 'GLASS'}

FIG_W, FIG_H = 8, 5
DPI = 300
TITLE_SIZE = 14; LABEL_SIZE = 13; TICK_SIZE = 11; ANNOT_SIZE = 9

dts = sorted(dtv.keys())
_fig_mask = globals().get('active_metric_mask', np.ones(len(test_samples), dtype=bool))


base_histories = globals().get(
    'base_ablation_histories',
    {k: v for k, v in ablation_histories.items() if not k.endswith('_gated_safe')}
)
gcm = globals().get('gate_calibration_metrics', {})
selected_base = gcm.get('base_best_name', best_name.replace('_gated_safe', ''))

fig, axes = plt.subplots(1, 2, figsize=(FIG_W+5, FIG_H), constrained_layout=True,
                         gridspec_kw={'width_ratios': [1.55, 1.0]})
ax = axes[0]
colors_abl = {'pos_only':'#8E44AD', 'pos_vel':'#16A085',
              'pos_vel_rr':'#C0392B', 'pos_vel_dop':'#F39C12',
              'pos_vel_rr_dop':'#2C3E50'}
for name, hist in base_histories.items():
    vals = np.asarray(hist.get('val', []), dtype=float)
    if vals.size == 0:
        continue
    epochs = np.arange(1, vals.size + 1)
    is_selected = (name == selected_base)
    ax.plot(epochs, vals,
            label=f'{name}' + (' (selected base)' if is_selected else ''),
            color=colors_abl.get(name, '#555555'),
            lw=2.8 if is_selected else 1.2,
            ls='-' if is_selected else ':',
            marker='o' if is_selected else 's',
            markersize=5 if is_selected else 3,
            alpha=1.0 if is_selected else 0.65)
ax.set_xlabel('Epoch', fontsize=LABEL_SIZE)
ax.set_ylabel('Validation loss (scale-normalized)', fontsize=LABEL_SIZE)
ax.set_yscale('log'); ax.grid(alpha=0.25, which='both')
ax.tick_params(labelsize=TICK_SIZE); ax.legend(fontsize=TICK_SIZE-1)

ax = axes[1]
gh = gcm.get('history', {})
gate_epochs = np.arange(1, len(gh.get('val', [])) + 1)
if len(gate_epochs):
    ax.plot(gate_epochs, gh.get('train', []), 'o-', color='#7F8C8D',
            lw=2.0, label='Train loss', alpha=0.8)
    ax.plot(gate_epochs, gh.get('val', []), 'o-', color='#8E44AD',
            lw=2.4, label='Val loss')
    ax.set_xlabel('Gate epoch', fontsize=LABEL_SIZE)
    ax.set_ylabel('Calibration loss', fontsize=LABEL_SIZE, color='#34495E')
    ax.tick_params(axis='y', labelcolor='#34495E', labelsize=TICK_SIZE)
    ax.tick_params(axis='x', labelsize=TICK_SIZE)
    ax.grid(alpha=0.25)

    ax2 = ax.twinx()
    if 'alpha_pos' in gh and 'alpha_vel' in gh:
        ax2.plot(gate_epochs, gh['alpha_pos'], 's--', color='#E67E22',
                 lw=1.6, label=r'$\alpha_{pos}$')
        ax2.plot(gate_epochs, gh['alpha_vel'], '^--', color=C_ML,
                 lw=1.6, label=r'$\alpha_{vel}$')
        ax2.set_ylabel('Learned gate value', fontsize=LABEL_SIZE, color='#2C3E50')
        ax2.tick_params(axis='y', labelcolor='#2C3E50', labelsize=TICK_SIZE)
        ax2.set_ylim(0.75, 1.02)

    base_m = gcm.get('base_metrics', {})
    gated_m = gcm.get('gated_metrics', {})
    if base_m and gated_m:
        def pct_delta(metric):
            return (base_m[metric] - gated_m[metric]) / base_m[metric] * 100.0
        txt = (f'Gated vs. base ({selected_base}):\n'
               f'pos:   {pct_delta("pos_mae"):+.1f}%\n'
               f'delay: {pct_delta("delay_mae"):+.1f}%\n'
               f'Doppler: {pct_delta("dop_mae"):+.1f}%')
        ax.text(0.04, 0.06, txt, transform=ax.transAxes, fontsize=ANNOT_SIZE,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.9))

    lines, labels = ax.get_legend_handles_labels()
    if 'alpha_pos' in gh and 'alpha_vel' in gh:
        l2, lab2 = ax2.get_legend_handles_labels()
        lines += l2; labels += lab2
    ax.legend(lines, labels, fontsize=TICK_SIZE-2, loc='center right')
else:
    ax.text(0.5, 0.5, 'Gate history not available', ha='center', va='center',
            transform=ax.transAxes)

fig.savefig('results/01_training_curve.png', dpi=DPI)
fig.savefig('results/01_training_curve.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 01_training_curve.png and 01_training_curve.pdf')


fig, ax = plt.subplots(figsize=(FIG_W+1, FIG_H), constrained_layout=True)
_budget_hz = NTN_3GPP['nr_prach_doppler_budget_hz']
for p in ['vanilla', 'residual', 'alongtrack', 'ml_corrected']:
    vals = np.asarray(horizon_budget_curves[p]['p95_budget_pct'], dtype=float)
    yerr = None
    
    if '_block_bootstrap_ci' in globals() and 'block_keys' in globals():
        los = []
        his = []
        err_pct = np.asarray(dop_stats['n256_DL'][p]['errs'], dtype=float) / _budget_hz * 100.0
        for dt, val in zip(dts, vals):
            mask_dt = (meta['dt_min'] == dt) & _fig_mask
            ci = _block_bootstrap_ci(err_pct, mask_dt, lambda x: np.percentile(x, 95))
            if np.isfinite(ci.get('lo', np.nan)) and np.isfinite(ci.get('hi', np.nan)) and np.isfinite(val):
                los.append(max(0.0, val - ci['lo']))
                his.append(max(0.0, ci['hi'] - val))
            else:
                los.append(np.nan)
                his.append(np.nan)
        if np.all(np.isfinite(los)) and np.all(np.isfinite(his)):
            yerr = np.vstack([los, his])
    fmt = 'o-' if p == 'ml_corrected' else 's--'
    ax.errorbar(dts, vals, yerr=yerr, fmt=fmt,
                color=PRED_COLORS[p], lw=2.5 if p == 'ml_corrected' else 1.5,
                markersize=8 if p == 'ml_corrected' else 6,
                capsize=3 if yerr is not None else 0,
                elinewidth=1.0 if yerr is not None else None,
                label=PRED_LABELS[p], alpha=1.0 if p == 'ml_corrected' else 0.8)
ax.axhline(100, color=C_BUDG, ls=':', lw=1.5, alpha=0.8)
ax.text(np.sqrt(dts[0] * dts[-1]), 102, '100% of NR-NTN budget', color=C_BUDG, fontsize=ANNOT_SIZE, ha='center', va='bottom', bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.2))
ax.set_xscale('log')
ax.set_xlabel('Propagation horizon (minutes)', fontsize=LABEL_SIZE)
ax.set_ylabel('P95 Doppler error / NR-NTN budget (%)', fontsize=LABEL_SIZE)
ax.legend(fontsize=TICK_SIZE, loc='upper left')
ax.tick_params(labelsize=TICK_SIZE); ax.grid(alpha=0.2, which='both')
ax.set_ylim(bottom=0)
fig.savefig('results/02_budget_consumption_vs_horizon.png', dpi=DPI)
fig.savefig('results/02_budget_consumption_vs_horizon.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 02_budget_consumption_vs_horizon.png and 02_budget_consumption_vs_horizon.pdf')


fig, axes = plt.subplots(1, 2, figsize=(FIG_W+4, FIG_H), constrained_layout=True)
finite_vals = np.concatenate([heatmap_van[np.isfinite(heatmap_van)], heatmap[np.isfinite(heatmap)]])
vmax = max(10.0, float(np.max(finite_vals))*1.05) if finite_vals.size else 100.0
cmap = LinearSegmentedColormap.from_list('budget', ['#27AE60', '#F4D03F', '#C0392B'])
for ax, data, title in [
    (axes[0], heatmap_van, 'Vanilla SGP4'),
    (axes[1], heatmap,     'GLASS'),
]:
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(dts))); ax.set_xticklabels([f'{d}' for d in dts], rotation=0)
    ax.set_yticks(range(len(age_bins_lb)))
    ax.set_yticklabels([l for _,_,l in age_bins_lb])
    ax.set_xlabel('Propagation horizon (min)', fontsize=LABEL_SIZE-1)
    ax.set_ylabel('Gap since previous TLE update', fontsize=LABEL_SIZE-1)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i,j]
            if not np.isnan(v):
                color = 'white' if v > 0.7 * vmax else 'black'
                n = int(heatmap_counts[i, j]) if 'heatmap_counts' in globals() else 0
                label = f'{v:.1f}\nN={n}' if n else f'{v:.1f}'
                ax.text(j, i, label, ha='center', va='center',
                        fontsize=ANNOT_SIZE-2, color=color)
fig.colorbar(im, ax=axes, label='P95 Doppler error / NR-NTN budget (%)', shrink=0.82)
for ax in axes:
    ax.text(0.01, -0.18, 'Each cell shows P95% and sample count N', transform=ax.transAxes,
            fontsize=ANNOT_SIZE-1, ha='left', va='top')
fig.savefig('results/03_budget_consumption_heatmap.png', dpi=DPI)
fig.savefig('results/03_budget_consumption_heatmap.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 03_budget_consumption_heatmap.png and 03_budget_consumption_heatmap.pdf')


fig, axes = plt.subplots(1, 2, figsize=(FIG_W+2, FIG_H), constrained_layout=True, sharey=True)
for ax, (cname, f_hz) in zip(axes, [('n255 DL', carriers['n255_DL']),
                                     ('n256 DL', carriers['n256_DL'])]):
    van_dt = []; mod_dt = []
    for dt in dts:
        mask = (meta['dt_min'] == dt) & _fig_mask
        if mask.sum() < 5:
            van_dt.append(np.nan); mod_dt.append(np.nan); continue
        key = cname.replace(' ', '_')
        van_dt.append(float(np.mean(dop_stats[key]['vanilla']['errs'][mask])))
        mod_dt.append(float(np.mean(dop_stats[key]['ml_corrected']['errs'][mask])))
    ax.plot(dts, van_dt, 'o-', color=C_VAN, lw=2, label='Vanilla SGP4')
    ax.plot(dts, mod_dt, 's-', color=C_ML,  lw=2, label='GLASS')
    ax.axhline(NTN_3GPP['nr_prach_doppler_budget_hz'], color=C_BUDG, ls=':', lw=1.2,
               label=f'NR-NTN budget ({NTN_3GPP["nr_prach_doppler_budget_hz"]:.0f} Hz)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Propagation horizon (min)', fontsize=LABEL_SIZE-1)
    ax.grid(alpha=0.2, which='both'); ax.legend(fontsize=TICK_SIZE-1)
axes[0].set_ylabel('Doppler MAE (Hz)', fontsize=LABEL_SIZE)
fig.savefig('results/04_doppler_per_band.png', dpi=DPI)
fig.savefig('results/04_doppler_per_band.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 04_doppler_per_band.png and 04_doppler_per_band.pdf')


fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=True)
for p in ['vanilla', 'residual', 'alongtrack', 'ml_corrected']:
    err = ta_stats[p]['errs'][_fig_mask]
    sorted_err = np.sort(err)
    cdf = np.linspace(0, 100, len(sorted_err))
    ax.plot(sorted_err, cdf, color=PRED_COLORS[p],
            lw=2.5 if p == 'ml_corrected' else 1.5,
            alpha=1.0 if p == 'ml_corrected' else 0.75,
            label=PRED_LABELS[p])
cp_us = NTN_3GPP['nr_cp_normal_15khz_sec'] * 1e6
ax.axvline(cp_us, color=C_BUDG, ls=':', lw=1.5,
           label=f'NR CP (15 kHz normal) = {cp_us:.2f} us')
ax.set_xlabel('Timing-advance error (us)', fontsize=LABEL_SIZE)
ax.set_ylabel('CDF (%)', fontsize=LABEL_SIZE)
ax.set_xscale('log'); ax.grid(alpha=0.2, which='both')
ax.legend(fontsize=TICK_SIZE); ax.tick_params(labelsize=TICK_SIZE)
ax.set_ylim(0, 105)
fig.savefig('results/05_timing_advance_cdf.png', dpi=DPI)
fig.savefig('results/05_timing_advance_cdf.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 05_timing_advance_cdf.png and 05_timing_advance_cdf.pdf')


fig, ax = plt.subplots(figsize=(FIG_W+1, FIG_H), constrained_layout=True)
configs = list(ablations.keys())
metric_keys_plot = ['dop_mae', 'delay_mae', 'pos_mae']
metric_names_plot = ['Doppler', 'Delay', 'Position']
metric_colors = ['#2471A3', '#16A085', '#C0392B']
x = np.arange(len(configs)); w = 0.25
for i, (k, nm, c) in enumerate(zip(metric_keys_plot, metric_names_plot, metric_colors)):
    vals = [(rv[k] - ablation_results[cf][k]) / rv[k] * 100 for cf in configs]
    ax.bar(x + (i-1)*w, vals, w, label=nm, color=c, edgecolor='black', lw=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(configs, rotation=15, fontsize=TICK_SIZE-1)
ax.set_ylabel('Improvement over Vanilla SGP4 (%)', fontsize=LABEL_SIZE)
ax.legend(fontsize=TICK_SIZE); ax.grid(axis='y', alpha=0.2)
fig.savefig('results/06_ablation.png', dpi=DPI)
fig.savefig('results/06_ablation.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 06_ablation.png and 06_ablation.pdf')


fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=True)
pct_labels = ['P50', 'P90', 'P95', 'P99']
pct_values = [50, 90, 95, 99]
van_pcts = [np.percentile(van_errs['pos'], q) for q in pct_values]
mod_pcts = [np.percentile(mod_errs['pos'], q) for q in pct_values]
x = np.arange(len(pct_labels)); w = 0.32
ax.bar(x - w/2, van_pcts, w, label='Vanilla SGP4', color=C_VAN, edgecolor='black', lw=0.5)
ax.bar(x + w/2, mod_pcts, w, label='GLASS', color=C_ML, edgecolor='black', lw=0.5)
for xi, (vv, mm) in enumerate(zip(van_pcts, mod_pcts)):
    ax.text(xi - w/2, vv+0.02, f'{vv:.3f}', ha='center', fontsize=ANNOT_SIZE, color=C_VAN, fontweight='bold')
    ax.text(xi + w/2, mm+0.02, f'{mm:.3f}', ha='center', fontsize=ANNOT_SIZE, color=C_ML, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(pct_labels, fontsize=TICK_SIZE)
ax.set_xlabel('Percentile', fontsize=LABEL_SIZE)
ax.set_ylabel('Position error (km)', fontsize=LABEL_SIZE)
ax.legend(fontsize=TICK_SIZE); ax.grid(axis='y', alpha=0.2)
fig.savefig('results/07_position_percentiles.png', dpi=DPI)
fig.savefig('results/07_position_percentiles.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 07_position_percentiles.png and 07_position_percentiles.pdf')


fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=True)
sat_list = sorted([s for s in SAT_IDX if any(sp['sat_name'] == s for sp in test_samples)])
van_ds = []; mod_ds = []
for sn in sat_list:
    mask = np.array([s == sn for s in meta['sat']]) & _fig_mask
    if mask.sum() == 0:
        van_ds.append(np.nan); mod_ds.append(np.nan); continue
    van_ds.append(float(np.mean(dop_stats['n256_DL']['vanilla']['errs'][mask])))
    mod_ds.append(float(np.mean(dop_stats['n256_DL']['ml_corrected']['errs'][mask])))
x = np.arange(len(sat_list)); w = 0.35
ax.bar(x - w/2, van_ds, w, label='Vanilla SGP4', color=C_VAN, edgecolor='black', lw=0.5)
ax.bar(x + w/2, mod_ds, w, label='GLASS', color=C_ML, edgecolor='black', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(sat_list, rotation=15, fontsize=TICK_SIZE-1)
ax.set_ylabel('Doppler MAE (Hz) at n256 DL', fontsize=LABEL_SIZE)
ax.legend(fontsize=TICK_SIZE); ax.grid(axis='y', alpha=0.2)
fig.savefig('results/08_per_satellite_doppler.png', dpi=DPI)
fig.savefig('results/08_per_satellite_doppler.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show(); print('Saved 08_per_satellite_doppler.png and 08_per_satellite_doppler.pdf')

print('\nAll 8 publication figures saved to results/ as PNG and Type-42 PDF')
print('Headline figure (preferred for LaTeX): 02_budget_consumption_vs_horizon.pdf')

In [ ]:


import shutil, subprocess, glob, os

figure_pdfs = sorted(glob.glob('results/*.pdf'))
if not figure_pdfs:
    print('No PDF figures found. Run the publication-figures cell first.')
elif shutil.which('pdffonts') is None:
    print('pdffonts not found. Optional check skipped.')
    print('To enable in Colab: !apt-get update -qq && apt-get install -y -qq poppler-utils')
else:
    bad = []
    for path in figure_pdfs:
        proc = subprocess.run(['pdffonts', path], capture_output=True, text=True)
        txt = proc.stdout.lower()
        has_type3 = 'type 3' in txt or 'type3' in txt
        if has_type3:
            bad.append(path)
        print(f"{path}: {'TYPE 3 FOUND' if has_type3 else 'OK'}")
    if bad:
        print('WARNING: Type 3 fonts found in:', bad)
    else:
        print('All checked figure PDFs appear free of Type 3 fonts.')

In [ ]:


import hashlib
from collections import defaultdict
try:
    from sgp4.exporter import export_tle
    HAS_EXPORTER = True
except ImportError:
    HAS_EXPORTER = False

model.eval()


def _geo_ecef_np(lat_d, lon_d, alt_k):
    # WGS84 geodetic observer position, matching the training/evaluation cells.
    return geodetic_to_ecef_wgs84_np(np.array([lat_d]), np.array([lon_d]), np.array([alt_k]))[0]

def _ntn_np(pos, vel, obs_ecef, lat_d, lon_d):
    """Scalar NTN metrics using the same WGS84 geodetic ENU basis as evaluation."""
    d = pos - obs_ecef; sr = max(np.linalg.norm(d), 1e-3)
    dn = d / sr
    east, north, up = enu_basis_from_geodetic_np(np.array([lat_d]), np.array([lon_d]))
    east, north, up = east[0], north[0], up[0]
    e = float(np.dot(d, east)); n = float(np.dot(d, north)); u_ = float(np.dot(d, up))
    horiz = max(math.sqrt(e*e + n*n), 1e-9)
    elev = math.degrees(math.atan2(u_, horiz))
    rr = float(np.dot(dn, vel)); dop = -rr/cfg.c_km_s*cfg.f_dl
    delay = sr/cfg.c_km_s*1e6
    ed = float(np.dot(vel, east)); nd = float(np.dot(vel, north)); ud = float(np.dot(vel, up))
    horiz_dot = (e*ed + n*nd) / horiz
    rate = ((horiz*ud - u_*horiz_dot) / max(sr*sr, 1e-12)) * math.degrees(1.0)
    return {'elev': elev, 'dop': dop, 'delay': delay, 'rate': rate}

@torch.no_grad()
def _run(mdl, sp, sv, dt_min, update_gap_h, sat_idx):
    """v8: no observer input; satellite index and update-gap proxy required."""
    pn = torch.tensor(sp/cfg.r_norm, dtype=torch.float32).unsqueeze(0).to(device)
    vn = torch.tensor(sv/cfg.v_norm, dtype=torch.float32).unsqueeze(0).to(device)
    pt = torch.tensor([dt_min/cfg.t_norm], dtype=torch.float32).to(device)
    ta = torch.tensor([min(update_gap_h, cfg.age_norm)/cfg.age_norm], dtype=torch.float32).to(device)
    pr = torch.tensor(sp, dtype=torch.float32).unsqueeze(0).to(device)
    vr = torch.tensor(sv, dtype=torch.float32).unsqueeze(0).to(device)
    sidx = torch.tensor([int(sat_idx)], dtype=torch.long).to(device)
    pred = mdl(pn, vn, pt, ta, sidx, pr, vr).cpu().numpy()[0]
    return pred[:3], pred[3:]

# ── Test bookkeeping ─────────────────────────────────────────────────
_tp = 0; _tf = 0; _results = []
def _report(group, name, passed, detail):
    global _tp, _tf
    passed = bool(passed)
    if passed: _tp += 1
    else: _tf += 1
    _results.append({'group': group, 'name': name, 'passed': passed, 'detail': str(detail)})
    print(f"  {'OK' if passed else 'FAIL'} {name}")
    if detail and not passed: print(f"    -> {detail}")

def _eval_samples(samples):
    """Run model on samples. Returns {metric: {'van': [...], 'mod': [...]}}."""
    out = {m: {'van': [], 'mod': []} for m in ['pos','dop','elev','delay','rate']}
    for s in samples:
        sp, sv = s['sgp4_pos'], s['sgp4_vel']
        s_tp, s_tv = s['true_pos'], s['true_vel']
        mp, mv = _run(model, sp, sv, s['prop_time_min'], s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)), s['sat_idx'])
        oe = _geo_ecef_np(s['obs_lat'], s['obs_lon'], s['obs_alt'])
        nv = _ntn_np(sp, sv, oe, s['obs_lat'], s['obs_lon']); nm = _ntn_np(mp, mv, oe, s['obs_lat'], s['obs_lon']); nt = _ntn_np(s_tp, s_tv, oe, s['obs_lat'], s['obs_lon'])
        out['pos']['van'].append(float(np.linalg.norm(sp - s_tp)))
        out['pos']['mod'].append(float(np.linalg.norm(mp - s_tp)))
        for mkey in ['dop','elev','delay','rate']:
            out[mkey]['van'].append(float(abs(nv[mkey] - nt[mkey])))
            out[mkey]['mod'].append(float(abs(nm[mkey] - nt[mkey])))
    return out

def _print_metric_table(bins_data, bin_labels, pass_metrics=['pos']):
    metrics = [('pos','Pos(km)'), ('dop','Dop(Hz)'), ('delay','Dly(us)'),
               ('elev','Elv(deg)'), ('rate','Rate(d/s)')]
    any_degraded = False
    header = f'  {"Bin":>10s} {"N":>6s}'
    for _, mn in metrics: header += f' {"V-"+mn:>10s} {"M-"+mn:>10s} {"Impr":>7s}'
    print(header); print('  ' + '-'*(len(header)-2))
    for label in bin_labels:
        d = bins_data.get(label)
        if d is None or not d['pos']['van']: continue
        n = len(d['pos']['van'])
        row = f'  {label:>10s} {n:>6d}'
        for mkey, _ in metrics:
            vm = np.mean(d[mkey]['van']); mm = np.mean(d[mkey]['mod'])
            imp = (vm - mm)/vm * 100 if vm > 0 else 0
            row += f' {vm:>10.4f} {mm:>10.4f} {imp:>+6.1f}%'
            if mkey in pass_metrics and mm > vm * 1.01: any_degraded = True
        print(row)
    return not any_degraded

OBS = [
    ('Arlington_VA',      38.8977, -77.0365, 0.0),
    ('Equatorial_0E',      0.0,      0.0,    0.0),
    ('Tromso_Norway',     69.6496,  18.9560, 0.0),
    ('Sydney_Australia', -33.8688, 151.2093, 0.0),
]

print('=' * 78)
print('  GLASS VERIFICATION TEST SUITE (v8)')
print('=' * 78)

# ── TEST 1: Zero-init property ───────────────────────────────────────
print('\n-- TEST 1: ZERO-INIT PROPERTY --')
print('  Freshly initialized CorrectionNetV3 base residual must output exactly SGP4; the v8 gate is calibrated post hoc.\n')
_zinit = CorrectionNetV3(N_SATS, cfg.hidden, cfg.layers, cfg.sat_embed_dim).to(device)
_zinit.eval()
_z_maxp = 0.0; _z_maxv = 0.0
for s in test_samples[:20]:
    mp, mv = _run(_zinit, s['sgp4_pos'], s['sgp4_vel'], s['prop_time_min'],
                  s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)), s['sat_idx'])
    _z_maxp = max(_z_maxp, float(np.max(np.abs(mp - s['sgp4_pos']))))
    _z_maxv = max(_z_maxv, float(np.max(np.abs(mv - s['sgp4_vel']))))
_report('ZERO_INIT', 'position_passthrough', _z_maxp < 1e-6, f'max dp = {_z_maxp:.2e} km')
_report('ZERO_INIT', 'velocity_passthrough', _z_maxv < 1e-6, f'max dv = {_z_maxv:.2e} km/s')
del _zinit

# Pick a Swarm satellite for end-to-end tests (Swarm-C has good POD coverage)
# Historical fix retained: key is correctly 'Swarm-C', not just 'C'
SWARM_KEY = 'Swarm-C' if 'Swarm-C' in tle_by_sat else list(tle_by_sat.keys())[0]
tles_c  = tle_by_sat[SWARM_KEY]
interp_c = interps[SWARM_KEY]
sidx_c   = SAT_IDX[SWARM_KEY]
print(f'\nUsing {SWARM_KEY} (idx={sidx_c}) for end-to-end tests')

# ── TEST 2: End-to-end reproducibility ────────────────────────────────
print('\n-- TEST 2: END-TO-END REPRODUCIBILITY --')
_n_tles = min(15, len(tles_c))
_sel_tles = [(int(i), tles_c[i]) for i in np.linspace(0, len(tles_c)-1, _n_tles, dtype=int)]
vcases = []; _skip = 0
for ti, (orig_idx, tle) in enumerate(_sel_tles):
    prev = [t['epoch_jd'] for t in tles_c if t['epoch_jd'] < tle['epoch_jd']]
    update_gap_h = (tle['epoch_jd'] - max(prev))*24 if prev else 0.0
    tle_lines = None
    if HAS_EXPORTER:
        try:
            l1, l2 = export_tle(tle['sat']); tle_lines = [l1.strip(), l2.strip()]
        except: pass
    for dt_min in [10, 60, 120, 240, 360]:
        tjd = tle['epoch_jd'] + dt_min/1440.0
        sp, sv = sgp4_ecef(tle['sat'], tjd)
        if sp is None: _skip += 1; continue
        t2_tp, t2_tv = interp_c(tjd)
        if t2_tp is None: _skip += 1; continue
        for oname, olat, olon, oalt in OBS:
            oe = _geo_ecef_np(olat, olon, oalt)
            mp, mv = _run(model, sp, sv, dt_min, update_gap_h, sidx_c)
            pe_van = float(np.linalg.norm(sp - t2_tp))
            pe_mod = float(np.linalg.norm(mp - t2_tp))
            case = {
                'case_id': f'{SWARM_KEY}_{ti:02d}_dt{dt_min}_{oname}',
                'tle_index': int(orig_idx),
                'tle_epoch_iso': tle['epoch_dt'].isoformat(),
                'tle_epoch_jd': round(float(tle['epoch_jd']), 10),
                'inter_update_gap_hours': round(float(update_gap_h), 4),
                'tle_age_hours': round(float(update_gap_h), 4),  # legacy alias
                'tle_propagation_age_hours': round(float(dt_min)/60.0, 4),
                'prop_time_min': int(dt_min), 'sat_idx': int(sidx_c),
                'observer': {'name': oname, 'lat': float(olat), 'lon': float(olon), 'alt': float(oalt)},
                'sgp4':  {'pos': [round(float(x),8) for x in sp],  'vel': [round(float(x),8) for x in sv]},
                'model': {'pos': [round(float(x),8) for x in mp],  'vel': [round(float(x),8) for x in mv]},
                'truth': {'pos': [round(float(x),8) for x in t2_tp],'vel': [round(float(x),8) for x in t2_tv]},
                'ntn_vanilla': {k: round(float(v),8) for k,v in _ntn_np(sp, sv, oe, olat, olon).items()},
                'ntn_model':   {k: round(float(v),8) for k,v in _ntn_np(mp, mv, oe, olat, olon).items()},
                'ntn_truth':   {k: round(float(v),8) for k,v in _ntn_np(t2_tp, t2_tv, oe, olat, olon).items()},
                'pos_err_km':  {'vanilla': round(pe_van,8), 'model': round(pe_mod,8)},
            }
            if tle_lines: case['tle_lines'] = tle_lines
            vcases.append(case)

print(f'  Generated {len(vcases)} verification cases ({_skip} skipped)')
_n_better = sum(1 for c in vcases if c['pos_err_km']['model'] < c['pos_err_km']['vanilla'])
_pct = _n_better / len(vcases) * 100 if vcases else 0
_report('REPRO', f'model_beats_vanilla_{_pct:.0f}pct', _pct > 50,
        f'{_n_better}/{len(vcases)} cases improved')
_vp = np.mean([c['pos_err_km']['vanilla'] for c in vcases])
_mp = np.mean([c['pos_err_km']['model'] for c in vcases])
_report('REPRO', 'position_mae_improved', _mp < _vp,
        f'vanilla={_vp:.4f} -> model={_mp:.4f} km ({(_vp-_mp)/_vp*100:+.1f}%)')
_vd = np.mean([abs(c['ntn_vanilla']['dop'] - c['ntn_truth']['dop']) for c in vcases])
_md = np.mean([abs(c['ntn_model']['dop']   - c['ntn_truth']['dop']) for c in vcases])
_report('REPRO', 'doppler_mae_improved', _md < _vd,
        f'vanilla={_vd:.2f} -> model={_md:.2f} Hz ({(_vd-_md)/_vd*100:+.1f}%)')

# ── TEST 3: Pareto per-case ──────────────────────────────────────────
print('\n-- TEST 3: PARETO VALIDITY PER-CASE --')
_t3 = _eval_samples(test_samples[:5000]); _t3n = len(_t3['pos']['van'])
for mkey, mname in [('pos','Position'), ('dop','Doppler')]:
    _imp = sum(1 for v,m in zip(_t3[mkey]['van'], _t3[mkey]['mod']) if m < v * 0.999)
    _deg = sum(1 for v,m in zip(_t3[mkey]['van'], _t3[mkey]['mod']) if m > v * 1.001)
    _pct_deg = _deg / _t3n * 100
    print(f'  {mname}: {_imp} improved, {_deg} degraded of {_t3n}')
    thresh = 10 if mkey == 'pos' else 50
    _report('PARETO', f'{mkey}_deg_{_pct_deg:.1f}pct', _pct_deg < thresh,
            f'{_pct_deg:.1f}% degraded (threshold {thresh}%)')

# ── TEST 4: Observer latitude stratification ──────────────────────────
print('\n-- TEST 4: OBSERVER LATITUDE STRATIFICATION --')
_lat_bins = [(-90,-45,'S-high'),(-45,-15,'S-mid'),(-15,15,'Equat'),(15,45,'N-mid'),(45,90,'N-high')]
_t4_binned = {l: [] for _,_,l in _lat_bins}
for s in test_samples:
    for lo, hi, l in _lat_bins:
        if lo <= s['obs_lat'] < hi: _t4_binned[l].append(s); break
_t4_data = {l: _eval_samples(v) for l, v in _t4_binned.items() if v}
_t4_ok = _print_metric_table(_t4_data, [l for _,_,l in _lat_bins], ['pos'])
_report('OBSERVER', 'pos_no_degradation_any_lat_bin', _t4_ok, '')

# ── TEST 5: Propagation time stratification ──────────────────────────
print('\n-- TEST 5: PROPAGATION TIME STRATIFICATION --')
_t5_binned = defaultdict(list)
for s in test_samples: _t5_binned[s['dt_min']].append(s)
_t5_data = {f'{dt}min': _eval_samples(v) for dt, v in sorted(_t5_binned.items())}
_t5_ok = _print_metric_table(_t5_data, [f'{dt}min' for dt in sorted(_t5_binned.keys())], ['pos'])
_report('PROP_TIME', 'pos_no_degradation_any_dt', _t5_ok, '')

# ── TEST 6: TLE age stratification ───────────────────────────────────
print('\n-- TEST 6: TLE AGE STRATIFICATION --')
_age_bins = [(0,2,'0-2h'),(2,6,'2-6h'),(6,12,'6-12h'),(12,24,'12-24h'),(24,9999,'24h+')]
_t6_binned = {l: [] for _,_,l in _age_bins}
for s in test_samples:
    for lo, hi, l in _age_bins:
        if lo <= s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)) < hi: _t6_binned[l].append(s); break
_t6_data = {l: _eval_samples(v) for l, v in _t6_binned.items() if v}
_t6_ok = _print_metric_table(_t6_data, [l for _,_,l in _age_bins], ['pos'])
_report('TLE_AGE', 'pos_no_degradation_any_age_bin', _t6_ok, '')

# ── TEST 7: Physical plausibility (per-sat bounds) ───────────────────
print('\n-- TEST 7: PHYSICAL PLAUSIBILITY --')
_violations = 0; _corr_ratios = []
for s in test_samples[:2000]:
    mp, mv = _run(model, s['sgp4_pos'], s['sgp4_vel'],
                  s['prop_time_min'], s.get('inter_update_gap_hours', s.get('tle_age_hours', 0.0)), s['sat_idx'])
    alt = np.linalg.norm(mp) - cfg.re; vel_mag = np.linalg.norm(mv)
    exp_alt = SATELLITES[s['sat_name']]['alt_km']
    if abs(alt - exp_alt) > 50: _violations += 1
    if not (7.3 < vel_mag < 7.8): _violations += 1
    sgp4_err = np.linalg.norm(s['sgp4_pos'] - s['true_pos'])
    if sgp4_err > 0.01:
        _corr_ratios.append(float(np.linalg.norm(mp - s['sgp4_pos']) / sgp4_err))
print(f'  Per-sat altitude+velocity violations: {_violations}/4000 checks')
_report('PHYSICAL', 'no_gross_violations', _violations < 20, f'{_violations} violations')
if _corr_ratios:
    _max_ratio = float(np.max(_corr_ratios))
    _report('PHYSICAL', 'correction_proportional', _max_ratio < 5.0,
            f'max correction/error = {_max_ratio:.2f}')

# ── TEST 8: SGP4 error magnitude stratification ──────────────────────
print('\n-- TEST 8: SGP4 ERROR MAGNITUDE STRATIFICATION --')
_van_errs = [float(np.linalg.norm(s['sgp4_pos']-s['true_pos'])) for s in test_samples]
_err_bins = [(0,0.3,'<0.3km'),(0.3,0.6,'0.3-0.6'),(0.6,1.0,'0.6-1.0'),(1.0,1.5,'1.0-1.5'),(1.5,99,'>1.5km')]
_t8_binned = {l: [] for _,_,l in _err_bins}
for s, ve in zip(test_samples, _van_errs):
    for lo, hi, l in _err_bins:
        if lo <= ve < hi: _t8_binned[l].append(s); break
_t8_data = {l: _eval_samples(v) for l, v in _t8_binned.items() if v}
_t8_ok = _print_metric_table(_t8_data, [l for _,_,l in _err_bins], ['pos'])
_report('ERR_MAG', 'pos_no_degradation_any_error_bin', _t8_ok, '')

# ── SUMMARY ──────────────────────────────────────────────────────────
print('\n' + '=' * 78)
print(f'  SUMMARY: {_tp}/{_tp+_tf} passed, {_tf}/{_tp+_tf} failed')
print('=' * 78)
if _tf > 0:
    print(f'\n  Failed:')
    for r in _results:
        if not r['passed']: print(f"    [{r['group']}] {r['name']}: {r['detail']}")

# Save verification cases for the reproducibility cell
verification_cases = {
    'metadata': {
        'model_version': 'v7',
        'best_ablation': best_name,
        'n_cases': len(vcases),
        'sat_idx_map': SAT_IDX,
        'tolerances': {'pos_km': 1e-4, 'vel_kms': 1e-6, 'dop_hz': 1e-2},
        'sha256_cases': hashlib.sha256(json.dumps(vcases, sort_keys=True).encode()).hexdigest(),
    },
    'cases': vcases,
}
with open('results/verification_cases.json', 'w') as f:
    json.dump(verification_cases, f, indent=2)
print(f'\nSaved results/verification_cases.json ({len(vcases)} cases)')

In [ ]:


import json, math, hashlib
import numpy as np
import torch, torch.nn as nn

# ── Model definitions matching v8 checkpoint ──
class _BaseVNet(nn.Module):
    def __init__(self, n_sats, hidden=48, layers=3, sat_embed_dim=4,
                 r_norm=6878.0, v_norm=7.5):
        super().__init__()
        self.r_norm, self.v_norm = r_norm, v_norm
        self.sat_embed = nn.Embedding(n_sats, sat_embed_dim)
        in_dim = 3 + 3 + 1 + 1 + sat_embed_dim
        net = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(layers-1): net.extend([nn.Linear(hidden, hidden), nn.Tanh()])
        self.net = nn.Sequential(*net)
        self.ph = nn.Linear(hidden, 3); self.vh = nn.Linear(hidden, 3)
        nn.init.zeros_(self.ph.weight); nn.init.zeros_(self.ph.bias)
        nn.init.zeros_(self.vh.weight); nn.init.zeros_(self.vh.bias)
    def forward(self, pn, vn, pt, ta, sidx, pr, vr):
        e = self.sat_embed(sidx)
        x = torch.cat([pn, vn, pt.unsqueeze(-1), ta.unsqueeze(-1), e], dim=-1)
        h = self.net(x)
        return torch.cat([pr + self.ph(h)*self.r_norm,
                          vr + self.vh(h)*self.v_norm], dim=-1)

class _GatedSafeWrapper(nn.Module):
    def __init__(self, base_model, n_sats, hidden=24, sat_embed_dim=4,
                 min_pos_alpha=0.0, min_vel_alpha=0.25,
                 max_pos_correction_km=2.5, max_vel_correction_kms=0.012):
        super().__init__()
        self.base = base_model
        self.min_pos_alpha = float(min_pos_alpha)
        self.min_vel_alpha = float(min_vel_alpha)
        self.max_pos_correction_km = float(max_pos_correction_km)
        self.max_vel_correction_kms = float(max_vel_correction_kms)
        self.sat_embed = nn.Embedding(n_sats, sat_embed_dim)
        in_dim = 3 + 3 + 1 + 1 + sat_embed_dim + 4
        self.gate = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
    @staticmethod
    def _clip_vec(v, max_norm):
        n = torch.norm(v, dim=-1, keepdim=True).clamp(min=1e-12)
        scale = torch.clamp(float(max_norm) / n, max=1.0)
        return v * scale
    def forward(self, pn, vn, pt, ta, sidx, pr, vr):
        base_pred = self.base(pn, vn, pt, ta, sidx, pr, vr)
        dp_raw = base_pred[:, :3] - pr
        dv_raw = base_pred[:, 3:] - vr
        dp = self._clip_vec(dp_raw, self.max_pos_correction_km)
        dv = self._clip_vec(dv_raw, self.max_vel_correction_kms)
        dp_norm = torch.norm(dp_raw, dim=-1, keepdim=True) / max(self.max_pos_correction_km, 1e-9)
        dv_norm = torch.norm(dv_raw, dim=-1, keepdim=True) / max(self.max_vel_correction_kms, 1e-12)
        dp_clip_ratio = torch.norm(dp, dim=-1, keepdim=True) / torch.norm(dp_raw, dim=-1, keepdim=True).clamp(min=1e-12)
        dv_clip_ratio = torch.norm(dv, dim=-1, keepdim=True) / torch.norm(dv_raw, dim=-1, keepdim=True).clamp(min=1e-12)
        e = self.sat_embed(sidx)
        gx = torch.cat([pn, vn, pt.unsqueeze(-1), ta.unsqueeze(-1), e,
                        dp_norm, dv_norm, dp_clip_ratio, dv_clip_ratio], dim=-1)
        raw_alpha = torch.sigmoid(self.gate(gx))
        alpha_pos = self.min_pos_alpha + (1.0 - self.min_pos_alpha) * raw_alpha[:, 0:1]
        alpha_vel = self.min_vel_alpha + (1.0 - self.min_vel_alpha) * raw_alpha[:, 1:2]
        return torch.cat([pr + alpha_pos * dp,
                          vr + alpha_vel * dv], dim=-1)

_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_ckpt = torch.load('results/ntn_model_final.pt', map_location=_dev, weights_only=False)
_norms = _ckpt.get('normalization', {})
_arch = _ckpt.get('architecture', '')
_gcfg = _ckpt.get('gate_config', {}) or {}

_base = _BaseVNet(n_sats=_ckpt['n_sats'],
                  hidden=_ckpt['hidden_dim'], layers=_ckpt['n_layers'],
                  sat_embed_dim=_ckpt['sat_embed_dim'],
                  r_norm=_norms.get('r_norm', 6878.0),
                  v_norm=_norms.get('v_norm', 7.5))
if _arch == 'GatedSafeWrapper':
    _mdl = _GatedSafeWrapper(
        _base, n_sats=_ckpt['n_sats'], hidden=_gcfg.get('gate_hidden', 24),
        sat_embed_dim=_ckpt['sat_embed_dim'],
        min_pos_alpha=_gcfg.get('gate_min_pos_alpha', 0.0),
        min_vel_alpha=_gcfg.get('gate_min_vel_alpha', 0.25),
        max_pos_correction_km=_gcfg.get('max_pos_correction_km', 2.5),
        max_vel_correction_kms=_gcfg.get('max_vel_correction_kms', 0.012),
    )
else:
    _mdl = _base
_mdl.load_state_dict(_ckpt['model_state_dict']); _mdl.eval().to(_dev)

with open('results/verification_cases.json') as f:
    _vdata = json.load(f)
_meta = _vdata['metadata']; _cases = _vdata['cases']
_tols = _meta.get('tolerances', {})

print('=' * 72)
print('  CHECKPOINT REPRODUCIBILITY CHECK (v8)')
print('=' * 72)
print(f'  Device:       {_dev}')
print(f'  Architecture: {_ckpt.get("architecture","?")} | '
      f'{_ckpt["hidden_dim"]}h x {_ckpt["n_layers"]}L | '
      f'{sum(p.numel() for p in _mdl.parameters())} params')
print(f'  Best config:  {_ckpt.get("best_ablation","?")} {_ckpt.get("best_loss_cfg","?")}')
print(f'  Cases:        {len(_cases)}')

_hash = hashlib.sha256(json.dumps(_cases, sort_keys=True).encode()).hexdigest()
_stored = _meta.get('sha256_cases', '')
print(f'  Integrity:    {"SHA-256 MATCH" if _hash == _stored else "SHA-256 MISMATCH"}')

print(f'\n  Running {len(_cases)} cases...')
print('-' * 72)
_ok = 0; _fail = 0; _fail_ids = []
_pos_tol = _tols.get('pos_km', 1e-4)
_vel_tol = _tols.get('vel_kms', 1e-6)

for _i, _c in enumerate(_cases):
    sp = np.array(_c['sgp4']['pos'], dtype=np.float32)
    sv = np.array(_c['sgp4']['vel'], dtype=np.float32)
    dt_min = _c['prop_time_min']; age_h = _c.get('inter_update_gap_hours', _c.get('tle_age_hours', 0.0))
    sidx = int(_c['sat_idx'])

    _r_norm = _norms.get('r_norm', 6878.0); _v_norm = _norms.get('v_norm', 7.5)
    _t_norm = _norms.get('t_norm', 1440.0); _a_norm = _norms.get('age_norm', 168.0)

    with torch.no_grad():
        pn = torch.tensor(sp/_r_norm, dtype=torch.float32).unsqueeze(0).to(_dev)
        vn = torch.tensor(sv/_v_norm, dtype=torch.float32).unsqueeze(0).to(_dev)
        pt = torch.tensor([dt_min/_t_norm], dtype=torch.float32).to(_dev)
        ta = torch.tensor([min(age_h, _a_norm)/_a_norm], dtype=torch.float32).to(_dev)
        pr = torch.tensor(sp, dtype=torch.float32).unsqueeze(0).to(_dev)
        vr = torch.tensor(sv, dtype=torch.float32).unsqueeze(0).to(_dev)
        sidx_t = torch.tensor([sidx], dtype=torch.long).to(_dev)
        pred = _mdl(pn, vn, pt, ta, sidx_t, pr, vr).cpu().numpy()[0]
    mp, mv = pred[:3], pred[3:]

    ref_mp = np.array(_c['model']['pos'])
    ref_mv = np.array(_c['model']['vel'])
    pos_err = float(np.max(np.abs(mp - ref_mp)))
    vel_err = float(np.max(np.abs(mv - ref_mv)))

    passed = pos_err < _pos_tol and vel_err < _vel_tol
    if passed: _ok += 1
    else:
        _fail += 1; _fail_ids.append(_c['case_id'])
        if _fail <= 3:
            print(f'  FAIL {_c["case_id"]}: pos_err={pos_err:.2e}, vel_err={vel_err:.2e}')

    if (_i+1) % 50 == 0 or _i == len(_cases)-1:
        print(f'  {_i+1}/{len(_cases)}: {_ok} ok, {_fail} fail')

print('-' * 72)
if _fail == 0:
    print(f'  ALL {len(_cases)} CASES REPRODUCE within tolerance')
    print(f'    pos tol: {_pos_tol:.0e} km    vel tol: {_vel_tol:.0e} km/s')
else:
    print(f'  {_fail}/{len(_cases)} cases FAILED to reproduce')
    if len(_fail_ids) <= 10:
        for fid in _fail_ids: print(f'    - {fid}')